# Reporte final de investigación

**Optimización de carteras multi-activo con crypto**

Este documento presenta la síntesis final del estudio en formato de research note aplicada. Consume únicamente outputs ya generados y auditados; no entrena modelos, no rehace pipelines pesados y excluye por diseño cualquier artefacto de `outputs/chapter5_debug/` como evidencia.

**Nota metodológica.** Salvo que se indique lo contrario, todas las métricas y figuras usan la metodología final auditada del proyecto, con calendario de días hábiles, retorno tipo drifted buy-and-hold, costes explícitos y comparación fuera de muestra sobre ventanas homogéneas.

**Convención de nombres.** En esta nota, `baseline final` se refiere a la especificación auditada definitiva del proyecto. La comparación histórica previa al endurecimiento metodológico queda relegada a trazabilidad secundaria y no forma parte del bloque visual principal.

In [580]:
# Phase 2B — imports, paths and helper utilities.
# Kept intentionally lightweight: no model training, no plotting yet.
from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Project paths -----------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CHAPTER5_DIR = OUTPUTS_DIR / "chapter5"
ROBUSTNESS_DIR = DATA_DIR / "robustness"
TAIL_RISK_DIR = DATA_DIR / "tail_risk"
REGIME_DIR = DATA_DIR / "regime_analysis"

# Explicitly excluded from final evidence (Phase 2 rule).
CHAPTER5_DEBUG_DIR = OUTPUTS_DIR / "chapter5_debug"

assert DATA_DIR.exists(), f"Missing data dir: {DATA_DIR}"
assert OUTPUTS_DIR.exists(), f"Missing outputs dir: {OUTPUTS_DIR}"
print("Project root:", PROJECT_ROOT)
print("Chapter 5 dir present:", CHAPTER5_DIR.exists())
print("Chapter 5 DEBUG dir (will be excluded):", CHAPTER5_DEBUG_DIR.exists())

Project root: c:\Users\dframent\OneDrive - NTT DATA EMEAL\Desktop\ProyectosData\Multi_Asset_Portfolio_Optimization
Chapter 5 dir present: True
Chapter 5 DEBUG dir (will be excluded): True


### Setup — helpers (safe IO, formatting, performance utilities)

The helpers below are deliberately small and side-effect free so that later sections can load only the artifacts they need without a global state. If a file is missing, `safe_read_csv` / `safe_read_json` return `None` and emit a single warning, so the notebook degrades gracefully instead of failing silently.

In [581]:
# ---- Safe IO -----------------------------------------------------------
def safe_read_csv(path: Path | str, **kwargs: Any) -> pd.DataFrame | None:
    """Read a CSV file if it exists, otherwise warn and return None."""
    p = Path(path)
    if not p.exists():
        warnings.warn(f"[safe_read_csv] missing artifact: {p}")
        return None
    try:
        return pd.read_csv(p, **kwargs)
    except Exception as exc:  # noqa: BLE001 — surface, do not crash the notebook
        warnings.warn(f"[safe_read_csv] failed reading {p}: {exc}")
        return None


def safe_read_json(path: Path | str) -> dict | None:
    """Read a JSON file if it exists, otherwise warn and return None."""
    p = Path(path)
    if not p.exists():
        warnings.warn(f"[safe_read_json] missing artifact: {p}")
        return None
    try:
        with p.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[safe_read_json] failed reading {p}: {exc}")
        return None

In [586]:
# ---- Display-name mappings (paper-style, human-readable) ---------------
STRATEGY_DISPLAY_NAMES: dict[str, str] = {
    # Chapter 1 baseline / benchmarks
    "min_variance": "MinVar",
    "minvar_baseline_ch1": "MinVar",
    "minvar_baseline_corrected": "MinVar",
    "minvar_no_crypto_control": "MinVar sin crypto",
    "equal_weight": "Equal Weight",
    "sixty_forty": "60/40 tradicional",
    "fixed_small_crypto": "Fixed Small Crypto",
    "spy_only": "SPY Only",
    # Chapter 3 — tail risk
    "cvar_baseline": "CVaR",
    "cvar_no_crypto_control": "CVaR sin crypto",
    # Chapter 5 — supervised overlay
    "minvar_dynamic_crypto_cap_naive_vol": "Overlay riesgo simple",
    "minvar_dynamic_crypto_cap_ml_vol": "Overlay riesgo ML",
    "minvar_dynamic_crypto_cap_stress_probability": "Overlay estrés",
    "minvar_combined_overlay": "Overlay combinado",
}

METRIC_DISPLAY_NAMES: dict[str, str] = {
    "ann_return": "Retorno anual",
    "annualized_return": "Retorno anual",
    "ann_return_gross": "Retorno anual (bruto)",
    "ann_return_net": "Retorno anual (neto)",
    "ann_volatility": "Volatilidad anual",
    "annualized_volatility": "Volatilidad anual",
    "ann_volatility_gross": "Volatilidad anual (bruta)",
    "ann_volatility_net": "Volatilidad anual (neta)",
    "sharpe": "Sharpe",
    "sharpe_gross": "Sharpe (bruto)",
    "sharpe_net": "Sharpe (neto)",
    "sortino": "Sortino",
    "calmar": "Calmar",
    "max_drawdown": "Max drawdown",
    "max_drawdown_gross": "Max drawdown (bruto)",
    "max_drawdown_net": "Max drawdown (neto)",
    "expected_shortfall": "ES95",
    "expected_shortfall_gross": "ES95 (bruto)",
    "expected_shortfall_net": "ES95 (neto)",
    "es95": "ES95",
    "turnover": "Turnover",
    "turnover_one_way": "Turnover (one-way)",
    "avg_turnover": "Turnover medio",
    "total_costs": "Costes totales",
    "cumulative_cost": "Coste acumulado",
    "avg_crypto_weight": "Peso crypto medio",
    "crypto_avg_weight": "Peso crypto medio",
    "max_crypto_weight": "Peso crypto máximo",
    "crypto_max_weight": "Peso crypto máximo",
    "crypto_median_weight": "Peso crypto mediano",
    "median_crypto_weight": "Peso crypto mediano",
    "n_rebalances_crypto_gt_2pct": "Rebalanceos con crypto > 2%",
    "n_crypto_rebalances_gt_2pct": "Rebalanceos con crypto > 2%",
    "share_crypto_gt_2pct": "Proporción con crypto > 2%",
}

MODEL_DISPLAY_NAMES: dict[str, str] = {
    "naive_rolling_vol": "Volatilidad rolling simple",
    "ewma": "EWMA",
    "ridge": "Ridge",
    "elastic_net": "Elastic Net",
    "random_forest": "Random Forest",
    "logistic": "Logística",
    "calibrated_logistic": "Logística calibrada",
    "logistic_regression": "Logística",
    "random_forest_classifier": "Random Forest",
    "gradient_boosting_classifier": "Gradient Boosting",
}


def humanize_strategy(name: str) -> str:
    return STRATEGY_DISPLAY_NAMES.get(name, name)


def humanize_metric(name: str) -> str:
    return METRIC_DISPLAY_NAMES.get(name, name)


def humanize_model(name: str) -> str:
    return MODEL_DISPLAY_NAMES.get(name, name)

In [587]:
# ---- Formatting and performance utilities ------------------------------
def format_percent(value: float | None, decimals: int = 1) -> str:
    """Format a 0-1 value as a percentage string with a fixed number of decimals."""
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value) * 100:.{decimals}f}%"


def compute_wealth(returns: pd.Series, initial: float = 1.0) -> pd.Series:
    """Cumulative wealth from a daily simple-return series."""
    if returns is None or len(returns) == 0:
        return pd.Series(dtype=float)
    r = pd.Series(returns).astype(float).sort_index().fillna(0.0)
    return initial * (1.0 + r).cumprod()


def compute_drawdown(returns_or_wealth: pd.Series, *, is_wealth: bool = False) -> pd.Series:
    """Compute drawdown (<= 0) from either a return series or a wealth series."""
    if returns_or_wealth is None or len(returns_or_wealth) == 0:
        return pd.Series(dtype=float)
    wealth = (
        pd.Series(returns_or_wealth).astype(float).sort_index()
        if is_wealth
        else compute_wealth(returns_or_wealth)
    )
    running_max = wealth.cummax()
    return wealth / running_max - 1.0


def pick_row(df: pd.DataFrame | None, column: str, value: str) -> pd.Series:
    """Return the first matching row or an empty Series."""
    if df is None or df.empty or column not in df.columns:
        return pd.Series(dtype=float)
    match = df.loc[df[column] == value]
    return match.iloc[0] if not match.empty else pd.Series(dtype=float)


def metric_value(row: pd.Series, *candidates: str) -> float:
    """Return the first available numeric metric from a row."""
    for candidate in candidates:
        if candidate in row.index and pd.notna(row[candidate]):
            return float(row[candidate])
    return float("nan")

### Setup — load core outputs

Loads only the **final** artifacts that will feed the paper-style sections (hallazgos por bloque, Claim vs Evidence, Strategy Comparison, etc.). Anything under `outputs/chapter5_debug/` is excluded by construction.

Each call uses `safe_read_csv` / `safe_read_json`, so a missing file produces a warning and a `None` entry in `CORE_OUTPUTS` instead of breaking the notebook. The summary printed below documents what was actually available at notebook run time.

In [588]:
CORE_OUTPUTS: dict[str, Any] = {
    # Chapter 1
    "dataset_metadata": safe_read_json(DATA_DIR / "dataset_metadata.json"),
    "backtest_summary": safe_read_csv(DATA_DIR / "backtest_summary.csv"),
    "benchmark_summary": safe_read_csv(DATA_DIR / "benchmark_summary.csv"),
    "portfolio_returns": safe_read_csv(DATA_DIR / "portfolio_returns.csv", parse_dates=["Date"]),
    "weights_history": safe_read_csv(DATA_DIR / "weights_history.csv", parse_dates=["rebalance_date"]),
    "turnover_history": safe_read_csv(DATA_DIR / "turnover_history.csv", parse_dates=["rebalance_date"]),
    # Chapter 2 — robustness
    "robustness_common_family_net": safe_read_csv(
        ROBUSTNESS_DIR / "robustness_summary_common_family_net.csv"
    ),
    "confidence_summary": safe_read_csv(ROBUSTNESS_DIR / "confidence_summary.csv"),
    # Chapter 3 — tail risk
    "tail_risk_summary": safe_read_csv(TAIL_RISK_DIR / "tail_risk_summary.csv"),
    "tail_risk_summary_net": safe_read_csv(TAIL_RISK_DIR / "tail_risk_summary_net.csv"),
    "tail_risk_returns": safe_read_csv(TAIL_RISK_DIR / "tail_risk_returns.csv", parse_dates=["date"]),
    "tail_risk_weights_panel": safe_read_csv(
        TAIL_RISK_DIR / "tail_risk_weights_panel.csv", parse_dates=["rebalance_date"]
    ),
    # Chapter 4 — regimes
    "regime_model_summary": safe_read_csv(REGIME_DIR / "regime_model_summary.csv"),
    "regime_transition_matrix": safe_read_csv(REGIME_DIR / "regime_transition_matrix.csv"),
    "regime_conditional_perf_net": safe_read_csv(
        REGIME_DIR / "regime_conditional_performance_net.csv"
    ),
    "regime_crypto_exposure": safe_read_csv(REGIME_DIR / "regime_crypto_exposure.csv"),
    # Chapter 5 — supervised overlay (final, NOT debug)
    "chapter5_metadata": safe_read_json(CHAPTER5_DIR / "chapter5_metadata.json"),
    "model_scores": safe_read_csv(CHAPTER5_DIR / "model_scores.csv"),
    "model_selection": safe_read_csv(CHAPTER5_DIR / "model_selection.csv"),
    "overlay_backtest_summary": safe_read_csv(CHAPTER5_DIR / "overlay_backtest_summary.csv"),
    "overlay_daily_returns": safe_read_csv(
        CHAPTER5_DIR / "overlay_daily_returns.csv", parse_dates=["date"]
    ),
    "overlay_weights": safe_read_csv(
        CHAPTER5_DIR / "overlay_weights.csv", parse_dates=["rebalance_date"]
    ),
    "overlay_decisions": safe_read_csv(
        CHAPTER5_DIR / "overlay_decisions.csv", parse_dates=["date"]
    ),
    # Cross-chapter audit
    "audit_fix_comparison": safe_read_csv(OUTPUTS_DIR / "audit_fix_comparison.csv"),
    "audit_fix_metadata": safe_read_json(OUTPUTS_DIR / "audit_fix_metadata.json"),
}

# Quick load report
load_report_rows = []
for key, val in CORE_OUTPUTS.items():
    if val is None:
        load_report_rows.append((key, "missing", ""))
    elif isinstance(val, pd.DataFrame):
        load_report_rows.append((key, "ok", f"{val.shape[0]} x {val.shape[1]}"))
    elif isinstance(val, dict):
        load_report_rows.append((key, "ok", f"{len(val)} keys"))
    else:
        load_report_rows.append((key, "ok", type(val).__name__))

load_report = pd.DataFrame(load_report_rows, columns=["artifact", "status", "shape_or_size"])
load_report

,artifact,status,shape_or_size
0,dataset_metadata,ok,8 keys
1,backtest_summary,ok,4 x 9
2,benchmark_summary,ok,3 x 9
3,portfolio_returns,ok,1914 x 5
4,weights_history,ok,89 x 7
5,turnover_history,ok,89 x 6
6,robustness_common_family_net,ok,52 x 28
7,confidence_summary,ok,5 x 16
8,tail_risk_summary,ok,4 x 20
9,tail_risk_summary_net,ok,4 x 26


## 1. Síntesis del estudio

Esta nota evalúa si BTC y ETH aportan beneficios robustos de diversificación dentro de una cartera multi-activo optimizada con criterios de riesgo. La pregunta no es si crypto tuvo episodios favorables, sino si permitir BTC/ETH mejora de forma material, estable y fuera de muestra una cartera MinVar neta de costes frente a un control sin crypto y frente al 60/40 tradicional.

La evidencia principal favorece una lectura conservadora: MinVar mejora el perfil retorno-riesgo frente al 60/40, pero el control sin crypto queda muy cerca de MinVar. La exposición media a BTC/ETH es baja, aparece de forma intermitente y rara vez supera umbrales económicamente relevantes. Los bloques de robustez, CVaR, regímenes y overlay supervisado ayudan a tensionar la tesis, pero no convierten crypto en una asignación estructural defendible.

**Tesis de trabajo.** El valor robusto está en la construcción de cartera basada en riesgo; BTC/ETH quedan, como máximo, como exposición táctica de baja intensidad.

In [590]:
from IPython.display import Markdown, display

summary_points = [
    "El estudio usa outputs ya generados para sintetizar los bloques empíricos; este reporte no entrena modelos ni ejecuta pipelines pesados.",
    "La comparación central distingue tres efectos: construcción MinVar, permiso a BTC/ETH y capas posteriores de control de riesgo.",
    "MinVar mejora frente al 60/40 en la ventana fuera de muestra, pero MinVar sin crypto queda muy cerca de MinVar.",
    "La diferencia atribuible a crypto es pequeña y debe juzgarse junto con el intervalo bootstrap, costes, turnover y exposición efectiva.",
    "CVaR, regímenes y modelos supervisados aportan diagnósticos útiles, pero no cambian de forma material la conclusión principal.",
    "La conclusión no es universal: depende de la muestra, del universo de activos, de los costes asumidos y de la especificación de rebalanceo.",
]

display(Markdown("\n".join(f"- {point}" for point in summary_points)))

- El estudio usa outputs ya generados para sintetizar los bloques empíricos; este reporte no entrena modelos ni ejecuta pipelines pesados.
- La comparación central distingue tres efectos: construcción MinVar, permiso a BTC/ETH y capas posteriores de control de riesgo.
- MinVar mejora frente al 60/40 en la ventana fuera de muestra, pero MinVar sin crypto queda muy cerca de MinVar.
- La diferencia atribuible a crypto es pequeña y debe juzgarse junto con el intervalo bootstrap, costes, turnover y exposición efectiva.
- CVaR, regímenes y modelos supervisados aportan diagnósticos útiles, pero no cambian de forma material la conclusión principal.
- La conclusión no es universal: depende de la muestra, del universo de activos, de los costes asumidos y de la especificación de rebalanceo.

## 2. Pregunta de investigación y criterio de materialidad

**Pregunta principal.** ¿Aportan BTC y ETH beneficios robustos de diversificación estructural cuando se evalúan dentro de un framework fuera de muestra, risk-based, neto de costes, con control sin crypto, CVaR, regímenes y overlay supervisado?

**Criterio de interpretación.** No toda diferencia positiva es económicamente relevante. En esta nota se interpreta como débil cualquier mejora pequeña, inestable, con intervalo que cruza cero o asociada a una exposición media muy baja.

**Regla práctica usada en la lectura.** Una mejora atribuida a crypto tendría que ser material, robusta y acompañada de exposición económicamente relevante. Como referencias visuales se usan, cuando procede, Δ Sharpe ±0.05, Δ retorno anual ±1 p.p. y frecuencia de rebalanceos con crypto >2%.

In [591]:
definitions = pd.DataFrame([
    ["MinVar", "Cartera que minimiza la varianza esperada bajo restricciones de pesos."],
    ["CVaR", "Objetivo de optimización centrado en pérdidas extremas esperadas más allá de un umbral."],
    ["ES95", "Expected Shortfall al 95%; aquí se trata como magnitud de pérdida, por lo que menor es mejor."],
    ["Max drawdown", "Mayor caída acumulada desde un máximo previo; menor pérdida es mejor."],
    ["OOS", "Evaluación fuera de muestra: las decisiones se prueban en datos no usados para calibrar."],
    ["Bootstrap", "Remuestreo por bloques para aproximar incertidumbre de diferencias de desempeño."],
    ["Overlay", "Regla posterior al optimizador que reduce exposición cuando aparecen señales de riesgo."],
], columns=["Concepto", "Definición operativa"])

display(Markdown("### Definiciones breves"))
display(definitions.style.hide(axis="index"))

### Definiciones breves

Concepto,Definición operativa
MinVar,Cartera que minimiza la varianza esperada bajo restricciones de pesos.
CVaR,Objetivo de optimización centrado en pérdidas extremas esperadas más allá de un umbral.
ES95,"Expected Shortfall al 95%; aquí se trata como magnitud de pérdida, por lo que menor es mejor."
Max drawdown,Mayor caída acumulada desde un máximo previo; menor pérdida es mejor.
OOS,Evaluación fuera de muestra: las decisiones se prueban en datos no usados para calibrar.
Bootstrap,Remuestreo por bloques para aproximar incertidumbre de diferencias de desempeño.
Overlay,Regla posterior al optimizador que reduce exposición cuando aparecen señales de riesgo.


## 3. Metodología del estudio y trazabilidad empírica

Esta sección conecta cada conclusión con una implementación concreta. La intención no es documentar todo el código, sino dejar visible la cadena mínima: pregunta, módulo, script, output y conclusión contrastable.

El notebook consume outputs finales ya generados. No usa artefactos de depuración de Chapter 5 y no reentrena modelos.

In [ ]:
from IPython.display import Markdown, displaydataset_meta = CORE_OUTPUTS.get("dataset_metadata") or {}chapter5_meta = CORE_OUTPUTS.get("chapter5_metadata") or {}assumption_rows = [    ["Política de calendario", dataset_meta.get("calendar_policy", "n/d")],    ["Factor de anualización", dataset_meta.get("annualization_factor", chapter5_meta.get("annualization_factor", "n/d"))],    ["Método de holding", chapter5_meta.get("holding_return_method", "drifted_buy_and_hold")],    ["Validación OOS", chapter5_meta.get("validation", {}).get("method", "walk-forward")],    ["Ventana de entrenamiento", chapter5_meta.get("validation", {}).get("train_window", "n/d")],    ["Ventana de test", chapter5_meta.get("validation", {}).get("test_window", "n/d")],    ["Embargo", chapter5_meta.get("validation", {}).get("embargo_days", "n/d")],    ["Horizonte de datos", f"{dataset_meta.get('start_date', 'n/d')} a {dataset_meta.get('end_date', 'n/d')}"] if dataset_meta else ["Horizonte de datos", "n/d"],]assumptions = pd.DataFrame(assumption_rows, columns=["Supuesto", "Valor"])display(Markdown("### Supuestos de evaluación"))display(assumptions.style.hide(axis="index"))display(Markdown(    "**Lectura.** Estos supuestos fijan el marco común de comparación. "    "El mapa auditable por capítulos se presenta después de la Figura 1."))

## 4. Tabla 2 — Hallazgos por bloque

La tabla resume qué pregunta responde cada capítulo, qué método usa y qué lectura económica deja para la tesis final. No sustituye las figuras: las prepara.

### Criterio de lectura

Cada fila debe responder una pregunta económica concreta: no basta con reportar una métrica favorable. La lectura se apoya en métricas netas, comparación contra controles y cautela ante diferencias pequeñas.

In [593]:
backtest = CORE_OUTPUTS.get("backtest_summary")
bench = CORE_OUTPUTS.get("benchmark_summary")
conf = CORE_OUTPUTS.get("confidence_summary")
tail = CORE_OUTPUTS.get("tail_risk_summary_net")
regime = CORE_OUTPUTS.get("regime_conditional_perf_net")
model_sel = CORE_OUTPUTS.get("model_selection")
overlay = CORE_OUTPUTS.get("overlay_backtest_summary")

rows = []

mv = pick_row(backtest, "strategy", "min_variance")
sf = pick_row(bench, "benchmark", "sixty_forty")
if not mv.empty and not sf.empty:
    evidence = f"MinVar Sharpe {mv['sharpe']:.2f} vs 60/40 {sf['sharpe']:.2f}; retorno anual {mv['ann_return'] * 100:.1f}% vs {sf['ann_return'] * 100:.1f}%."
else:
    evidence = "Filas MinVar/60-40 no disponibles en los resúmenes cargados."
rows.append(["Baseline", "¿La construcción MinVar mejora al 60/40?", "Backtest OOS con costes y drifted buy-and-hold", evidence, "El beneficio principal procede de la construcción risk-based.", "Dependencia del periodo y del universo de activos."])

c1 = pd.DataFrame()
if conf is not None and not conf.empty and "comparison_id" in conf.columns:
    c1 = conf.loc[conf["comparison_id"] == "C1_anchor_pair"]
if not c1.empty:
    r = c1.iloc[0]
    evidence = f"Δ Sharpe crypto {r['point_estimate_difference']:+.3f}; IC [{r['ci_lower']:+.3f}, {r['ci_upper']:+.3f}]."
else:
    evidence = "Comparación bootstrap C1 no disponible."
rows.append(["Robustez", "¿El permiso a crypto es robusto?", "Control sin crypto y bootstrap por bloques", evidence, "Un punto estimado positivo no basta si el intervalo cruza cero.", "El bootstrap no elimina riesgo de especificación ni múltiples comparaciones."])

cvar = pick_row(tail, "strategy", "cvar_baseline")
minvar_tail = pick_row(tail, "strategy", "minvar_baseline_ch1")
if not cvar.empty and not minvar_tail.empty:
    evidence = f"CVaR Sharpe {cvar['sharpe_net']:.3f} vs MinVar {minvar_tail['sharpe_net']:.3f}; ES95 {cvar['expected_shortfall_net'] * 100:.2f}% vs {minvar_tail['expected_shortfall_net'] * 100:.2f}%."
else:
    evidence = "Filas CVaR/MinVar no disponibles."
rows.append(["CVaR y cola", "¿Optimizar cola cambia la conclusión?", "Minimización CVaR histórica y ventanas de stress", evidence, "CVaR sirve como prueba de cola, pero no domina de forma económica clara.", "ES95 y MaxDD requieren leer dirección de mejora con cuidado."])

mv_reg = regime.loc[regime["strategy"] == "minvar_baseline_ch1"] if regime is not None and not regime.empty else pd.DataFrame()
if not mv_reg.empty:
    low = mv_reg.loc[mv_reg["regime_name"] == "Low-stress / Risk-on", "sharpe"]
    high = mv_reg.loc[mv_reg["regime_name"] == "High-stress / Risk-off", "sharpe"]
    evidence = f"Sharpe MinVar {float(low.iloc[0]):.2f} en riesgo contenido vs {float(high.iloc[0]):.2f} en estrés alto." if not low.empty and not high.empty else "Performance condicional incompleta."
else:
    evidence = "Tabla de performance por régimen no disponible."
rows.append(["Regímenes", "¿El contexto de mercado cambia la lectura?", "Features de régimen, clustering/estado y evaluación condicional", evidence, "Los regímenes son diagnóstico histórico; no validan una regla predictiva live.", "Resultados sensibles a definición de estados y muestra."])

if model_sel is not None and not model_sel.empty:
    improved = int(model_sel["improved_vs_naive"].fillna(False).sum())
    total = len(model_sel)
    evidence = f"{improved}/{total} targets seleccionan un modelo que mejora al naive."
else:
    evidence = "Selección de modelos no disponible."
rows.append(["Señales y overlay", "¿La señal supervisada mejora la cartera?", "Validación OOS de targets de riesgo y backtest de overlay", evidence, "Señal predictiva y valor económico son cosas distintas; el overlay debe juzgarse por métricas netas.", "Riesgo de selección de targets, calibración y transferencia temporal."])

findings_df = pd.DataFrame(rows, columns=["Bloque", "Pregunta", "Método", "Evidencia clave", "Lectura económica", "Limitación"])
display(findings_df.style.hide(axis="index"))

Bloque,Pregunta,Método,Evidencia clave,Lectura económica,Limitación
Baseline,¿La construcción MinVar mejora al 60/40?,Backtest OOS con costes y drifted buy-and-hold,MinVar Sharpe 1.10 vs 60/40 0.76; retorno anual 11.8% vs 9.4%.,El beneficio principal procede de la construcción risk-based.,Dependencia del periodo y del universo de activos.
Robustez,¿El permiso a crypto es robusto?,Control sin crypto y bootstrap por bloques,"Δ Sharpe crypto +0.007; IC [-0.024, +0.042].",Un punto estimado positivo no basta si el intervalo cruza cero.,El bootstrap no elimina riesgo de especificación ni múltiples comparaciones.
CVaR y cola,¿Optimizar cola cambia la conclusión?,Minimización CVaR histórica y ventanas de stress,CVaR Sharpe 1.095 vs MinVar 1.093; ES95 1.59% vs 1.55%.,"CVaR sirve como prueba de cola, pero no domina de forma económica clara.",ES95 y MaxDD requieren leer dirección de mejora con cuidado.
Regímenes,¿El contexto de mercado cambia la lectura?,"Features de régimen, clustering/estado y evaluación condicional",Sharpe MinVar 1.63 en riesgo contenido vs 0.15 en estrés alto.,Los regímenes son diagnóstico histórico; no validan una regla predictiva live.,Resultados sensibles a definición de estados y muestra.
Señales y overlay,¿La señal supervisada mejora la cartera?,Validación OOS de targets de riesgo y backtest de overlay,10/12 targets seleccionan un modelo que mejora al naive.,Señal predictiva y valor económico son cosas distintas; el overlay debe juzgarse por métricas netas.,"Riesgo de selección de targets, calibración y transferencia temporal."


**Lectura analítica.** La tabla muestra una estructura de falsación progresiva. Primero se valida si MinVar mejora al 60/40; después se pregunta si crypto explica esa mejora; finalmente se tensiona la conclusión con cola, regímenes y overlay supervisado. La evidencia se debilita justo donde el claim crypto tendría que fortalecerse.

## 5. Tabla 3 — Evidencia frente a hipótesis

Esta tabla funciona como control de overclaiming. Cada afirmación se juzga contra evidencia concreta, confianza y punto débil. El objetivo no es cerrar una verdad universal, sino identificar qué queda defendible con los outputs disponibles.

### Criterio de veredicto

Los veredictos distinguen entre evidencia favorable, evidencia insuficiente y evidencia contraria. Las diferencias pequeñas, los intervalos que cruzan cero y las exposiciones medias muy bajas reducen la fuerza del claim aunque el punto estimado tenga el signo esperado.

In [594]:
backtest = CORE_OUTPUTS.get("backtest_summary")
bench = CORE_OUTPUTS.get("benchmark_summary")
conf = CORE_OUTPUTS.get("confidence_summary")
tail = CORE_OUTPUTS.get("tail_risk_summary_net")
regime_perf = CORE_OUTPUTS.get("regime_conditional_perf_net")
overlay_summary = CORE_OUTPUTS.get("overlay_backtest_summary")
weights_history = CORE_OUTPUTS.get("weights_history")

baseline_row = pick_row(overlay_summary, "strategy", "minvar_baseline_corrected")
no_crypto_row = pick_row(overlay_summary, "strategy", "minvar_no_crypto_control")
overlay_row = pick_row(overlay_summary, "strategy", "minvar_combined_overlay")
bench_6040 = pick_row(bench, "benchmark", "sixty_forty")
backtest_minvar = pick_row(backtest, "strategy", "min_variance")

rows = []
if not bench_6040.empty and not backtest_minvar.empty:
    evidence = f"Sharpe {backtest_minvar['sharpe']:.2f} vs {bench_6040['sharpe']:.2f}; retorno anual {backtest_minvar['ann_return'] * 100:.1f}% vs {bench_6040['ann_return'] * 100:.1f}%."
else:
    evidence = "Comparación MinVar/60-40 no disponible."
rows.append(["MinVar mejora frente al 60/40.", evidence, "Sí, en esta muestra", "Alta", "No implica superioridad universal fuera del periodo evaluado."])

crypto_evidence = "Control sin crypto o bootstrap no disponible."
crypto_confidence = "Media"
if not baseline_row.empty and not no_crypto_row.empty:
    delta_sharpe = metric_value(baseline_row, "sharpe") - metric_value(no_crypto_row, "sharpe")
    crypto_evidence = f"Δ Sharpe {delta_sharpe:+.3f}."
    if conf is not None and not conf.empty:
        c1 = conf.loc[conf["comparison_id"] == "C1_anchor_pair"]
        if not c1.empty:
            r = c1.iloc[0]
            crypto_evidence = f"Δ Sharpe {delta_sharpe:+.3f}; IC [{r['ci_lower']:+.3f}, {r['ci_upper']:+.3f}]."
            crypto_confidence = "Alta" if bool(r["ci_includes_zero"]) else "Media"
rows.append(["Crypto explica la mejora.", crypto_evidence, "No defendible", crypto_confidence, "El efecto incremental es pequeño y el IC cruza cero."])

crypto_weight_evidence = "Panel de pesos no disponible."
if weights_history is not None and not weights_history.empty:
    crypto_cols = [col for col in ["BTC-USD", "ETH-USD"] if col in weights_history.columns]
    if crypto_cols:
        focus = weights_history[crypto_cols].sum(axis=1)
        crypto_weight_evidence = f"Peso medio {focus.mean() * 100:.2f}%; mediana {focus.median() * 100:.2f}%; >2% en {(focus > 0.02).mean() * 100:.1f}% de rebalanceos."
rows.append(["Crypto es asignación estructural.", crypto_weight_evidence, "No", "Alta", "Una asignación estructural debería ser frecuente y material."])
rows.append(["Crypto es exposición táctica.", crypto_weight_evidence, "Compatible", "Alta", "Táctica no significa necesariamente rentable o escalable."])

cvar_evidence = "Comparación CVaR-MinVar no disponible."
if tail is not None and not tail.empty:
    cvar_row = pick_row(tail, "strategy", "cvar_baseline")
    minvar_row = pick_row(tail, "strategy", "minvar_baseline_ch1")
    if not cvar_row.empty and not minvar_row.empty:
        cvar_evidence = f"Sharpe {metric_value(cvar_row, 'sharpe_net'):.3f} vs {metric_value(minvar_row, 'sharpe_net'):.3f}; ES95 {metric_value(cvar_row, 'expected_shortfall_net') * 100:.2f}% vs {metric_value(minvar_row, 'expected_shortfall_net') * 100:.2f}%."
rows.append(["CVaR domina.", cvar_evidence, "No claro", "Media", "Las diferencias de cola y retorno son pequeñas."])

regime_evidence = "Performance por régimen no disponible."
if regime_perf is not None and not regime_perf.empty:
    mv_reg = regime_perf.loc[regime_perf["strategy"] == "minvar_baseline_ch1"]
    low = mv_reg.loc[mv_reg["regime_name"] == "Low-stress / Risk-on", "sharpe"]
    high = mv_reg.loc[mv_reg["regime_name"] == "High-stress / Risk-off", "sharpe"]
    if not low.empty and not high.empty:
        regime_evidence = f"Sharpe MinVar {float(low.iloc[0]):.2f} en riesgo contenido vs {float(high.iloc[0]):.2f} en estrés alto."
rows.append(["Regímenes cambian la interpretación.", regime_evidence, "Sí, como diagnóstico", "Media", "No son una señal predictiva live validada."])

overlay_evidence = "Comparación overlay-MinVar no disponible."
if not baseline_row.empty and not overlay_row.empty:
    overlay_evidence = f"Δ Sharpe {metric_value(overlay_row, 'sharpe') - metric_value(baseline_row, 'sharpe'):+.3f}; Δ ES95 {(metric_value(overlay_row, 'es95') - metric_value(baseline_row, 'es95')) * 100:+.2f} p.p."
rows.append(["ML/overlay aporta valor económico.", overlay_evidence, "No robusto", "Media", "La señal predictiva no se traduce claramente en mejora neta de cartera."])

rows.append(["El framework incorpora controles robustos.", "Control sin crypto, costes, bootstrap, CVaR, regímenes y overlay disponibles.", "Sí, con cautela", "Alta", "Metodológicamente defendible no significa universal ni libre de riesgo de especificación."])

claim_df = pd.DataFrame(rows, columns=["Hipótesis / afirmación", "Evidencia", "Veredicto", "Confianza", "Punto débil"])
display(claim_df.style.hide(axis="index"))

Hipótesis / afirmación,Evidencia,Veredicto,Confianza,Punto débil
MinVar mejora frente al 60/40.,Sharpe 1.10 vs 0.76; retorno anual 11.8% vs 9.4%.,"Sí, en esta muestra",Alta,No implica superioridad universal fuera del periodo evaluado.
Crypto explica la mejora.,"Δ Sharpe +0.011; IC [-0.024, +0.042].",No defendible,Alta,El efecto incremental es pequeño y el IC cruza cero.
Crypto es asignación estructural.,Peso medio 0.53%; mediana 0.00%; >2% en 6.7% de rebalanceos.,No,Alta,Una asignación estructural debería ser frecuente y material.
Crypto es exposición táctica.,Peso medio 0.53%; mediana 0.00%; >2% en 6.7% de rebalanceos.,Compatible,Alta,Táctica no significa necesariamente rentable o escalable.
CVaR domina.,Sharpe 1.095 vs 1.093; ES95 1.59% vs 1.55%.,No claro,Media,Las diferencias de cola y retorno son pequeñas.
Regímenes cambian la interpretación.,Sharpe MinVar 1.63 en riesgo contenido vs 0.15 en estrés alto.,"Sí, como diagnóstico",Media,No son una señal predictiva live validada.
ML/overlay aporta valor económico.,Δ Sharpe -0.038; Δ ES95 -0.00 p.p.,No robusto,Media,La señal predictiva no se traduce claramente en mejora neta de cartera.
El framework incorpora controles robustos.,"Control sin crypto, costes, bootstrap, CVaR, regímenes y overlay disponibles.","Sí, con cautela",Alta,Metodológicamente defendible no significa universal ni libre de riesgo de especificación.


**Lectura analítica.** La tabla separa evidencia favorable de atribución causal. MinVar puede mejorar al 60/40 sin que BTC/ETH expliquen esa mejora. La conclusión central queda condicionada por materialidad, robustez estadística y exposición efectiva.

## 5.1. Tablas C y D — Estrategias finales y atribución económica

Antes del bloque visual, estas tablas separan dos niveles de lectura: desempeño/riesgo de cada estrategia e impacto incremental de cada capa metodológica. Esta separación evita atribuir a crypto lo que puede venir de la construcción MinVar.

In [595]:
def _fmt_pct_value(value: float, decimals: int = 1) -> str:
    if value is None or pd.isna(value):
        return "n/d"
    return f"{float(value) * 100:.{decimals}f}%"


def _fmt_delta_pct(value: float, decimals: int = 2) -> str:
    if value is None or pd.isna(value):
        return "n/d"
    return f"{float(value) * 100:+.{decimals}f} p.p."


def _fmt_delta(value: float, decimals: int = 3) -> str:
    if value is None or pd.isna(value):
        return "n/d"
    return f"{float(value):+.{decimals}f}"

bench = CORE_OUTPUTS.get("benchmark_summary")
overlay_summary = CORE_OUTPUTS.get("overlay_backtest_summary")
tail_net = CORE_OUTPUTS.get("tail_risk_summary_net")

bench_6040 = pick_row(bench, "benchmark", "sixty_forty")
baseline_row = pick_row(overlay_summary, "strategy", "minvar_baseline_corrected")
no_crypto_row = pick_row(overlay_summary, "strategy", "minvar_no_crypto_control")
overlay_combined_row = pick_row(overlay_summary, "strategy", "minvar_combined_overlay")

strategy_rows = []

def _append_strategy(label: str, row: pd.Series, source: str) -> None:
    if row is None or row.empty:
        return
    crypto_rebalances = metric_value(row, "n_rebalances_crypto_gt_2pct", "n_crypto_rebalances_gt_2pct")
    strategy_rows.append({
        "Estrategia": label,
        "Retorno anual": _fmt_pct_value(metric_value(row, "ann_return", "annualized_return", "ann_return_net")),
        "Volatilidad anual": _fmt_pct_value(metric_value(row, "ann_volatility", "annualized_volatility", "ann_volatility_net")),
        "Sharpe": f"{metric_value(row, 'sharpe', 'sharpe_net'):.3f}",
        "Max drawdown": _fmt_pct_value(metric_value(row, "max_drawdown", "max_drawdown_net")),
        "ES95": _fmt_pct_value(metric_value(row, "es95", "expected_shortfall_net")),
        "Turnover": _fmt_pct_value(metric_value(row, "avg_turnover", "turnover", "turnover_one_way")),
        "Costes": _fmt_pct_value(metric_value(row, "total_costs", "cumulative_cost"), decimals=2),
        "Peso crypto medio": _fmt_pct_value(metric_value(row, "avg_crypto_weight", "crypto_avg_weight"), decimals=2),
        "Peso crypto máximo": _fmt_pct_value(metric_value(row, "max_crypto_weight", "crypto_max_weight"), decimals=2),
        "Rebalances crypto >2%": int(crypto_rebalances) if pd.notna(crypto_rebalances) else "n/d",
        "Fuente": source,
    })

_append_strategy("60/40 tradicional", bench_6040, "benchmark_summary.csv")
_append_strategy("MinVar", baseline_row, "overlay_backtest_summary.csv")
_append_strategy("MinVar sin crypto", no_crypto_row, "overlay_backtest_summary.csv")
if tail_net is not None and not tail_net.empty:
    _append_strategy("CVaR", pick_row(tail_net, "strategy", "cvar_baseline"), "tail_risk_summary_net.csv")
    _append_strategy("CVaR sin crypto", pick_row(tail_net, "strategy", "cvar_no_crypto_control"), "tail_risk_summary_net.csv")
if overlay_summary is not None and not overlay_summary.empty:
    _append_strategy("Overlay riesgo simple", pick_row(overlay_summary, "strategy", "minvar_dynamic_crypto_cap_naive_vol"), "overlay_backtest_summary.csv")
    _append_strategy("Overlay riesgo ML", pick_row(overlay_summary, "strategy", "minvar_dynamic_crypto_cap_ml_vol"), "overlay_backtest_summary.csv")
    _append_strategy("Overlay combinado", overlay_combined_row, "overlay_backtest_summary.csv")

strategy_table = pd.DataFrame(strategy_rows)
perf_cols = ["Estrategia", "Retorno anual", "Volatilidad anual", "Sharpe", "Max drawdown", "ES95"]
impl_cols = ["Estrategia", "Turnover", "Costes", "Peso crypto medio", "Peso crypto máximo", "Rebalances crypto >2%", "Fuente"]

display(Markdown("### Tabla 4 — Comparación final de estrategias"))
display(strategy_table[perf_cols].style.hide(axis="index"))
display(Markdown("### Tabla 5 — Atribución económica e implementabilidad"))
display(strategy_table[impl_cols].style.hide(axis="index"))

def _delta_row(effect: str, comparison: str, left: pd.Series, right: pd.Series, interpretation: str) -> dict:
    left_return = metric_value(left, "ann_return", "annualized_return", "ann_return_net")
    right_return = metric_value(right, "ann_return", "annualized_return", "ann_return_net")
    left_sharpe = metric_value(left, "sharpe", "sharpe_net")
    right_sharpe = metric_value(right, "sharpe", "sharpe_net")
    left_dd = metric_value(left, "max_drawdown", "max_drawdown_net")
    right_dd = metric_value(right, "max_drawdown", "max_drawdown_net")
    left_es = metric_value(left, "es95", "expected_shortfall_net")
    right_es = metric_value(right, "es95", "expected_shortfall_net")
    return {
        "Efecto": effect,
        "Comparación": comparison,
        "Δ Retorno": _fmt_delta_pct(left_return - right_return if pd.notna(left_return) and pd.notna(right_return) else np.nan),
        "Δ Sharpe": _fmt_delta(left_sharpe - right_sharpe if pd.notna(left_sharpe) and pd.notna(right_sharpe) else np.nan),
        "Δ MaxDD": _fmt_delta_pct(left_dd - right_dd if pd.notna(left_dd) and pd.notna(right_dd) else np.nan),
        "Δ ES95": _fmt_delta_pct(left_es - right_es if pd.notna(left_es) and pd.notna(right_es) else np.nan),
        "Interpretación": interpretation,
    }

attrib_rows = []
if not baseline_row.empty and not bench_6040.empty:
    attrib_rows.append(_delta_row("Construcción MinVar", "MinVar vs 60/40", baseline_row, bench_6040, "Mejora asociada al framework de construcción de cartera."))
if not baseline_row.empty and not no_crypto_row.empty:
    attrib_rows.append(_delta_row("Permitir crypto", "MinVar vs MinVar sin crypto", baseline_row, no_crypto_row, "Diferencia pequeña; no basta para atribución estructural a BTC/ETH."))
if tail_net is not None and not tail_net.empty:
    cvar_row = pick_row(tail_net, "strategy", "cvar_baseline")
    minvar_tail = pick_row(tail_net, "strategy", "minvar_baseline_ch1")
    if not cvar_row.empty and not minvar_tail.empty:
        attrib_rows.append(_delta_row("Objetivo de cola", "CVaR vs MinVar", cvar_row, minvar_tail, "Prueba de cola; mejora solo si reduce pérdidas sin sacrificar perfil neto."))
if not overlay_combined_row.empty and not baseline_row.empty:
    attrib_rows.append(_delta_row("Overlay", "Overlay combinado vs MinVar", overlay_combined_row, baseline_row, "Control supervisado trazable, pero valor económico limitado si no mejora métricas netas."))
if overlay_summary is not None and not overlay_summary.empty:
    simple_row = pick_row(overlay_summary, "strategy", "minvar_dynamic_crypto_cap_naive_vol")
    ml_row = pick_row(overlay_summary, "strategy", "minvar_dynamic_crypto_cap_ml_vol")
    if not simple_row.empty and not ml_row.empty:
        attrib_rows.append(_delta_row("ML incremental", "Overlay ML vs overlay simple", ml_row, simple_row, "Aísla si ML añade algo frente a una regla simple de riesgo."))

attrib_table = pd.DataFrame(attrib_rows)
display(Markdown("### Tabla D. Atribución económica"))
display(attrib_table.style.hide(axis="index"))

### Tabla 4 — Comparación final de estrategias

Estrategia,Retorno anual,Volatilidad anual,Sharpe,Max drawdown,ES95
60/40 tradicional,9.4%,12.2%,0.764,-27.7%,n/d
MinVar,11.8%,10.8%,1.093,-23.4%,1.6%
MinVar sin crypto,11.6%,10.7%,1.082,-23.4%,1.5%
CVaR,12.0%,11.0%,1.095,-23.3%,1.6%
CVaR sin crypto,11.7%,10.9%,1.071,-23.2%,1.6%
Overlay riesgo simple,11.8%,10.8%,1.093,-23.4%,1.6%
Overlay riesgo ML,11.8%,10.8%,1.093,-23.4%,1.6%
Overlay combinado,11.4%,10.8%,1.055,-24.6%,1.6%


### Tabla 5 — Atribución económica e implementabilidad

Estrategia,Turnover,Costes,Peso crypto medio,Peso crypto máximo,Rebalances crypto >2%,Fuente
60/40 tradicional,n/d,n/d,n/d,n/d,n/d,benchmark_summary.csv
MinVar,2.7%,0.24%,0.53%,3.23%,6,overlay_backtest_summary.csv
MinVar sin crypto,2.6%,0.22%,0.00%,0.00%,0,overlay_backtest_summary.csv
CVaR,n/d,n/d,n/d,n/d,n/d,tail_risk_summary_net.csv
CVaR sin crypto,n/d,n/d,n/d,n/d,n/d,tail_risk_summary_net.csv
Overlay riesgo simple,2.7%,0.24%,0.53%,3.23%,6,overlay_backtest_summary.csv
Overlay riesgo ML,2.7%,0.24%,0.53%,3.23%,6,overlay_backtest_summary.csv
Overlay combinado,4.0%,0.35%,0.53%,3.23%,6,overlay_backtest_summary.csv


### Tabla D. Atribución económica

Efecto,Comparación,Δ Retorno,Δ Sharpe,Δ MaxDD,Δ ES95,Interpretación
Construcción MinVar,MinVar vs 60/40,+2.41 p.p.,+0.329,+4.23 p.p.,n/d,Mejora asociada al framework de construcción de cartera.
Permitir crypto,MinVar vs MinVar sin crypto,+0.17 p.p.,+0.011,-0.00 p.p.,+0.01 p.p.,Diferencia pequeña; no basta para atribución estructural a BTC/ETH.
Objetivo de cola,CVaR vs MinVar,+0.24 p.p.,+0.002,+0.17 p.p.,+0.03 p.p.,Prueba de cola; mejora solo si reduce pérdidas sin sacrificar perfil neto.
Overlay,Overlay combinado vs MinVar,-0.40 p.p.,-0.038,-1.19 p.p.,-0.00 p.p.,"Control supervisado trazable, pero valor económico limitado si no mejora métricas netas."
ML incremental,Overlay ML vs overlay simple,+0.00 p.p.,+0.000,+0.00 p.p.,+0.00 p.p.,Aísla si ML añade algo frente a una regla simple de riesgo.


## 4. Diseño empírico del contrasteEste tramo abre la evidencia final con la lógica de investigación del estudio. La Figura 1 ordena los contrastes por preguntas empíricas; la Tabla 1 traduce cada capítulo en código, scripts, outputs, resultado y aporte a la tesis final.### Figura 1 — Diseño empírico del contrasteLa figura resume la secuencia de pruebas usadas para responder si BTC/ETH aportan diversificación estructural a una cartera multi-activo risk-based.

In [ ]:
COLOR = {
    "minvar": "#1f4e79",
    "minvar_no_crypto": "#8a6f3d",
    "sixty_forty": "#6c757d",
    "equal_weight": "#9aa0a6",
    "fixed_crypto": "#7b5ea7",
    "cvar": "#7a3b69",
    "cvar_no_crypto": "#b07aa1",
    "overlay": "#2f7f7f",
    "overlay_simple": "#3d8b8b",
    "overlay_ml": "#1f6f8b",
    "overlay_stress": "#b86b3c",
    "risk_contained": "#4f8a5f",
    "risk_contained_light": "#dcebdc",
    "high_stress": "#b85c5c",
    "high_stress_light": "#f3d7d7",
    "positive": "#4f8a5f",
    "negative": "#b85c5c",
    "mixed": "#c9a646",
    "neutral": "#d9dee3",
    "threshold": "#555555",
    "zero": "#333333",
    "grid": "#e6e6e6",
    "background": "#f7f7f7",
}

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": COLOR["background"],
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

VIS_STRATEGY_NAMES = {
    "sixty_forty": "60/40 tradicional",
    "min_variance": "MinVar",
    "minvar_baseline_ch1": "MinVar",
    "minvar_baseline_corrected": "MinVar",
    "minvar_no_crypto_control": "MinVar sin crypto",
    "cvar_baseline": "CVaR",
    "cvar_no_crypto_control": "CVaR sin crypto",
    "minvar_dynamic_crypto_cap_naive_vol": "Overlay riesgo simple",
    "minvar_dynamic_crypto_cap_ml_vol": "Overlay riesgo ML",
    "minvar_dynamic_crypto_cap_stress_probability": "Overlay estrés",
    "minvar_combined_overlay": "Overlay combinado",
}

COLOR = {
	"minvar": "#1f4e79",
	"minvar_no_crypto": "#8a6f3d",
	"sixty_forty": "#6c757d",
	"equal_weight": "#9aa0a6",
	"fixed_crypto": "#7b5ea7",
	"cvar": "#7a3b69",
	"cvar_no_crypto": "#b07aa1",
	"overlay": "#2f7f7f",
	"overlay_simple": "#3d8b8b",
	"overlay_ml": "#1f6f8b",
	"overlay_stress": "#b86b3c",
	"risk_contained": "#4f8a5f",
	"risk_contained_light": "#dcebdc",
	"high_stress": "#b85c5c",
	"high_stress_light": "#f3d7d7",
	"positive": "#4f8a5f",
	"negative": "#b85c5c",
	"mixed": "#c9a646",
	"neutral": "#d9dee3",
	"threshold": "#555555",
	"zero": "#333333",
	"grid": "#e6e6e6",
	"background": "#f7f7f7",
}

VIS_COLORS = {
	"60/40 tradicional": COLOR["sixty_forty"],
	"MinVar": COLOR["minvar"],
	"MinVar sin crypto": COLOR["minvar_no_crypto"],
	"Equal Weight": COLOR["equal_weight"],
	"Fixed Small Crypto": COLOR["fixed_crypto"],
	"CVaR": COLOR["cvar"],
	"CVaR sin crypto": COLOR["cvar_no_crypto"],
	"Overlay riesgo simple": COLOR["overlay_simple"],
	"Overlay riesgo ML": COLOR["overlay_ml"],
	"Overlay estrés": COLOR["overlay_stress"],
	"Overlay combinado": COLOR["overlay"],
}

VIS_REGIME_NAMES = {
    "Low-stress / Risk-on": "Riesgo contenido",
    "High-stress / Risk-off": "Estrés alto",
}

def vis_h_strategy(name: str) -> str:
    return VIS_STRATEGY_NAMES.get(name, humanize_strategy(name).replace("without crypto", "sin crypto"))


def vis_h_regime(name: str) -> str:
    return VIS_REGIME_NAMES.get(name, name)


def vis_pct_axis(ax, axis: str = "y", decimals: int = 0) -> None:
    formatter = mtick.PercentFormatter(xmax=1.0, decimals=decimals)
    if axis == "x":
        ax.xaxis.set_major_formatter(formatter)
    else:
        ax.yaxis.set_major_formatter(formatter)


def vis_note(message: str) -> None:
    display(Markdown(f"> **Nota:** {message}"))


def vis_show_reading(que: str, obs: str, imp: str, lim: str) -> None:
    display(Markdown(
        f"**Qué muestra:** {que}  \\n"
        f"**Qué se observa:** {obs}  \\n"
        f"**Qué implica:** {imp}  \\n"
        f"**Limitación:** {lim}"
    ))


def vis_metric(row: pd.Series, *candidates: str) -> float:
    for candidate in candidates:
        if candidate in row.index and pd.notna(row[candidate]):
            return float(row[candidate])
    return float("nan")


def vis_pick_row(df: pd.DataFrame | None, column: str, value: str) -> pd.Series:
    if df is None or df.empty or column not in df.columns:
        return pd.Series(dtype=float)
    match = df.loc[df[column] == value]
    return match.iloc[0] if not match.empty else pd.Series(dtype=float)


def vis_load_output(key: str, fallback_path: Path, **kwargs) -> pd.DataFrame | None:
    value = CORE_OUTPUTS.get(key)
    if value is not None:
        return value
    return safe_read_csv(fallback_path, **kwargs)


def vis_build_series_map() -> dict[str, pd.Series]:
    series_map: dict[str, pd.Series] = {}

    portfolio_returns = CORE_OUTPUTS.get("portfolio_returns")
    if portfolio_returns is not None and {"Date", "sixty_forty", "min_variance"}.issubset(portfolio_returns.columns):
        tmp = portfolio_returns.copy()
        tmp["Date"] = pd.to_datetime(tmp["Date"])
        tmp = tmp.set_index("Date").sort_index()
        series_map["60/40 tradicional"] = tmp["sixty_forty"].astype(float)
        series_map["MinVar"] = tmp["min_variance"].astype(float)

    overlay_daily_returns = vis_load_output(
        "overlay_daily_returns",
        CHAPTER5_DIR / "overlay_daily_returns.csv",
        parse_dates=["date"],
    )
    if overlay_daily_returns is not None and {"date", "daily_return", "strategy"}.issubset(overlay_daily_returns.columns):
        tmp = overlay_daily_returns.copy()
        tmp["date"] = pd.to_datetime(tmp["date"])
        for strategy in ["minvar_baseline_corrected", "minvar_no_crypto_control", "minvar_combined_overlay"]:
            subset = tmp.loc[tmp["strategy"] == strategy, ["date", "daily_return"]]
            if not subset.empty:
                series_map[vis_h_strategy(strategy)] = subset.set_index("date")["daily_return"].astype(float).sort_index()

    tail_risk_returns = vis_load_output(
        "tail_risk_returns",
        TAIL_RISK_DIR / "tail_risk_returns.csv",
        parse_dates=["date"],
    )
    if tail_risk_returns is not None and {"date", "portfolio_return", "strategy"}.issubset(tail_risk_returns.columns):
        tmp = tail_risk_returns.copy()
        tmp["date"] = pd.to_datetime(tmp["date"])
        for strategy in ["cvar_baseline", "cvar_no_crypto_control"]:
            subset = tmp.loc[tmp["strategy"] == strategy, ["date", "portfolio_return"]]
            if not subset.empty:
                series_map[vis_h_strategy(strategy)] = subset.set_index("date")["portfolio_return"].astype(float).sort_index()

    return series_map


VIS_SERIES_MAP = vis_build_series_map()
VIS_COMMON_NAMES = ["60/40 tradicional", "MinVar", "MinVar sin crypto"]
VIS_COMMON_RETURNS = pd.concat(
    [VIS_SERIES_MAP[name].rename(name) for name in VIS_COMMON_NAMES if name in VIS_SERIES_MAP],
    axis=1,
    join="inner",
).dropna(how="any") if all(name in VIS_SERIES_MAP for name in VIS_COMMON_NAMES) else pd.DataFrame()
VIS_COMMON_WEALTH = VIS_COMMON_RETURNS.apply(compute_wealth) if not VIS_COMMON_RETURNS.empty else pd.DataFrame()
VIS_COMMON_DRAWDOWN = VIS_COMMON_RETURNS.apply(compute_drawdown) if not VIS_COMMON_RETURNS.empty else pd.DataFrame()

VIS_BACKTEST = CORE_OUTPUTS.get("backtest_summary")
VIS_BENCH = CORE_OUTPUTS.get("benchmark_summary")
VIS_OVERLAY_SUMMARY = vis_load_output("overlay_backtest_summary", CHAPTER5_DIR / "overlay_backtest_summary.csv")
VIS_OVERLAY_WEIGHTS = vis_load_output(
    "overlay_weights",
    CHAPTER5_DIR / "overlay_weights.csv",
    parse_dates=["rebalance_date"],
 )
VIS_OVERLAY_DECISIONS = vis_load_output(
    "overlay_decisions",
    CHAPTER5_DIR / "overlay_decisions.csv",
    parse_dates=["date"],
 )
VIS_WEIGHTS_HISTORY = CORE_OUTPUTS.get("weights_history")
VIS_TAIL_NET = vis_load_output("tail_risk_summary_net", TAIL_RISK_DIR / "tail_risk_summary_net.csv")
VIS_TAIL_WEIGHTS = vis_load_output(
    "tail_risk_weights_panel",
    TAIL_RISK_DIR / "tail_risk_weights_panel.csv",
    parse_dates=["rebalance_date"],
 )
VIS_STRESS = vis_load_output("stress_summary", TAIL_RISK_DIR / "stress_summary.csv")
VIS_CONFIDENCE = vis_load_output("confidence_summary", ROBUSTNESS_DIR / "confidence_summary.csv")
VIS_ROBUSTNESS = vis_load_output("robustness_common_family_net", ROBUSTNESS_DIR / "robustness_summary_common_family_net.csv")
VIS_REGIME_PERF = safe_read_csv(REGIME_DIR / "regime_conditional_performance_net.csv")
VIS_REGIME_CRYPTO = vis_load_output("regime_crypto_exposure", REGIME_DIR / "regime_crypto_exposure.csv")
VIS_REGIME_LABELS = safe_read_csv(REGIME_DIR / "regime_labels.csv", parse_dates=["date"])
VIS_MODEL_SCORES = vis_load_output("model_scores", CHAPTER5_DIR / "model_scores.csv")
VIS_MODEL_SELECTION = vis_load_output("model_selection", CHAPTER5_DIR / "model_selection.csv")
VIS_FORECAST_BUCKETS = vis_load_output("forecast_bucket_analysis", CHAPTER5_DIR / "forecast_bucket_analysis.csv")
VIS_CALIBRATION = vis_load_output("calibration_tables", CHAPTER5_DIR / "calibration_tables.csv")
VIS_AUDIT = vis_load_output("audit_fix_comparison", OUTPUTS_DIR / "audit_fix_comparison.csv")

VIS_BASELINE_ROW = vis_pick_row(VIS_OVERLAY_SUMMARY, "strategy", "minvar_baseline_corrected")
VIS_NO_CRYPTO_ROW = vis_pick_row(VIS_OVERLAY_SUMMARY, "strategy", "minvar_no_crypto_control")
VIS_OVERLAY_ROW = vis_pick_row(VIS_OVERLAY_SUMMARY, "strategy", "minvar_combined_overlay")

def vis_prepare_crypto_panel() -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    crypto_cols = ["BTC-USD", "ETH-USD"]

    if VIS_WEIGHTS_HISTORY is not None and not VIS_WEIGHTS_HISTORY.empty:
        tmp = VIS_WEIGHTS_HISTORY.copy()
        tmp["rebalance_date"] = pd.to_datetime(tmp["rebalance_date"])
        tmp["strategy"] = "minvar_baseline_corrected"
        tmp["crypto_total"] = tmp[[col for col in crypto_cols if col in tmp.columns]].sum(axis=1)
        frames.append(tmp[["rebalance_date", "strategy", "crypto_total"]])

    if VIS_OVERLAY_WEIGHTS is not None and not VIS_OVERLAY_WEIGHTS.empty:
        tmp = VIS_OVERLAY_WEIGHTS.copy()
        tmp["rebalance_date"] = pd.to_datetime(tmp["rebalance_date"])
        tmp["crypto_total"] = tmp[[col for col in crypto_cols if col in tmp.columns]].sum(axis=1)
        frames.append(tmp[["rebalance_date", "strategy", "crypto_total"]])

    if VIS_TAIL_WEIGHTS is not None and not VIS_TAIL_WEIGHTS.empty:
        tmp = VIS_TAIL_WEIGHTS.copy()
        tmp["rebalance_date"] = pd.to_datetime(tmp["rebalance_date"])
        tmp["crypto_total"] = tmp[[col for col in crypto_cols if col in tmp.columns]].sum(axis=1)
        frames.append(tmp[["rebalance_date", "strategy", "crypto_total"]])

    if not frames:
        return pd.DataFrame(columns=["rebalance_date", "strategy", "crypto_total", "Estrategia"])

    panel = pd.concat(frames, ignore_index=True)
    panel = panel.dropna(subset=["crypto_total"]).drop_duplicates(subset=["rebalance_date", "strategy"])
    panel["Estrategia"] = panel["strategy"].map(vis_h_strategy)
    return panel.sort_values(["Estrategia", "rebalance_date"])


VIS_CRYPTO_PANEL = vis_prepare_crypto_panel()

fig, ax = plt.subplots(figsize=(17, 10))ax.set_xlim(0, 1)ax.set_ylim(0, 1)ax.axis("off")ax.text(    0.5,    0.965,    "Figura 1 — Diseño empírico del contraste",    ha="center",    va="center",    fontsize=20,    weight="bold",    color=COLOR["minvar"],)ax.text(    0.5,    0.918,    "Pregunta central: ¿BTC/ETH aportan diversificación estructural a una cartera multi-activo risk-based?",    ha="center",    va="center",    fontsize=12.2,    color="#333333",)contrasts = [    ("1", "¿MinVar mejora al 60/40?", "Baseline walk-forward", "MinVar aporta frente al benchmark tradicional", COLOR["minvar"]),    ("2", "¿Crypto explica esa mejora?", "Control sin BTC/ETH", "El delta crypto es pequeño y no robusto", COLOR["minvar_no_crypto"]),    ("3", "¿La exposición BTC/ETH es estructural?", "Pesos y frecuencia de rebalanceos", "La exposición aparece baja e intermitente", COLOR["mixed"]),    ("4", "¿Sobrevive a robustez, costes y bootstrap?", "Sensibilidad y confianza estadística", "La evidencia no cumple materialidad robusta", COLOR["risk_contained"]),    ("5", "¿CVaR cambia la lectura?", "Pérdidas extremas y stress windows", "La cola añade diagnóstico; no rescata la tesis crypto", COLOR["cvar"]),    ("6", "¿Los regímenes alteran la interpretación?", "Performance condicional por estado", "El contexto importa, pero no valida timing live", COLOR["high_stress"]),    ("7", "¿Overlay supervisado aporta valor económico?", "Señales OOS y caps dinámicos", "La señal no monetiza robustamente frente a MinVar", COLOR["overlay"]),]x_left = 0.07x_mid = 0.49x_right = 0.76row_top = 0.82row_gap = 0.085box_h = 0.062ax.text(x_left, 0.865, "Secuencia de contrastes", ha="left", va="center", fontsize=11, weight="bold", color="#444444")ax.text(x_mid, 0.865, "Implementación empírica", ha="left", va="center", fontsize=11, weight="bold", color="#444444")ax.text(x_right, 0.865, "Lectura para la tesis", ha="left", va="center", fontsize=11, weight="bold", color="#444444")ax.plot([0.055, 0.945], [0.845, 0.845], color=COLOR["grid"], linewidth=2)for idx, (number, question, implementation, reading, color) in enumerate(contrasts):    y = row_top - idx * row_gap    ax.plot([0.105, 0.105], [y - 0.03, y + 0.03], color="#d0d5da", linewidth=2, zorder=1)    circle = plt.Circle((0.105, y), 0.026, color=color, ec="white", lw=1.4, zorder=3)    ax.add_patch(circle)    ax.text(0.105, y, number, ha="center", va="center", fontsize=10.5, weight="bold", color="white", zorder=4)    q_box = FancyBboxPatch((0.14, y - box_h / 2), 0.30, box_h, boxstyle="round,pad=0.012,rounding_size=0.014", linewidth=1.0, edgecolor=color, facecolor="white")    ax.add_patch(q_box)    ax.text(0.153, y, question, ha="left", va="center", fontsize=9.7, weight="bold", color="#222222")    impl_box = FancyBboxPatch((x_mid, y - box_h / 2), 0.22, box_h, boxstyle="round,pad=0.012,rounding_size=0.014", linewidth=0.9, edgecolor="#c7cdd3", facecolor="#f7f7f7")    ax.add_patch(impl_box)    ax.text(x_mid + 0.012, y, implementation, ha="left", va="center", fontsize=9.3, color="#333333")    verdict_box = FancyBboxPatch((x_right, y - box_h / 2), 0.18, box_h, boxstyle="round,pad=0.012,rounding_size=0.014", linewidth=1.0, edgecolor=color, facecolor="#ffffff")    ax.add_patch(verdict_box)    ax.text(x_right + 0.012, y, reading, ha="left", va="center", fontsize=8.8, color="#222222", wrap=True)    ax.add_patch(FancyArrowPatch((0.445, y), (x_mid - 0.012, y), arrowstyle="-|>", mutation_scale=10, lw=1.0, color="#7a7f85"))    ax.add_patch(FancyArrowPatch((x_mid + 0.225, y), (x_right - 0.012, y), arrowstyle="-|>", mutation_scale=10, lw=1.0, color="#7a7f85"))final_y = 0.12final_box = FancyBboxPatch((0.075, final_y - 0.035), 0.85, 0.11, boxstyle="round,pad=0.018,rounding_size=0.018", linewidth=1.2, edgecolor="#b87922", facecolor="#fdebd0")ax.add_patch(final_box)ax.text(0.10, final_y + 0.045, "Conclusión visual", ha="left", va="center", fontsize=11.2, weight="bold", color="#8a5a00")conclusion_items = [    "MinVar aporta frente al 60/40",    "Crypto no explica robustamente la mejora",    "Crypto aparece como exposición táctica/intermitente",    "CVaR, regímenes y overlay añaden diagnóstico, pero no cambian la tesis principal",]for idx, item in enumerate(conclusion_items):    ax.text(0.11 + (idx % 2) * 0.405, final_y + 0.012 - (idx // 2) * 0.042, f"• {item}", ha="left", va="center", fontsize=9.5, color="#333333")plt.tight_layout()plt.show()vis_show_reading(    "La Figura 1 presenta la lógica de contraste del estudio: primero se evalúa si MinVar aporta frente al 60/40; después se analiza si BTC/ETH explican esa mejora; finalmente se somete la hipótesis crypto a robustez, riesgo de cola, regímenes y overlay supervisado.",    "Cada capítulo añade una prueba distinta. El baseline identifica el valor de la construcción risk-based; la robustez y el control sin crypto aíslan el efecto de BTC/ETH; CVaR y regímenes revisan el riesgo desde otros ángulos; el overlay comprueba si señales predictivas pueden convertirse en valor económico.",    "La conclusión final no depende de una sola figura o métrica. Para defender crypto como asignación estructural haría falta un efecto material, robusto y acompañado de exposición persistente. La evidencia consolidada no cumple esas condiciones.",    "La tabla permite auditar cómo cada capítulo llega a sus outputs, pero no elimina las limitaciones de muestra, universo, datos y supuestos.",)

In [ ]:
from html import escapefrom IPython.display import HTML, Markdown, displaychapter_map = [    {"Bloque del estudio": "Capítulo 1 — Baseline MinVar", "Pregunta que responde": "¿Una cartera MinVar constrained mejora al 60/40 tradicional en la muestra evaluada?", "Qué se implementa": "Backtest walk-forward con rebalanceo mensual, pesos drifted buy-and-hold, restricciones long-only, límite por activo y límite crypto.", "Código principal": ["src/backtest.py", "src/optimizer.py", "src/metrics.py", "src/benchmarks.py"], "Scripts reproducibles": ["scripts/run_backtest.py", "scripts/run_benchmarks.py"], "Outputs principales": ["portfolio returns", "weights history", "backtest summary", "benchmark summary"], "Resultado principal": "MinVar mejora al 60/40 en perfil retorno-riesgo dentro de la muestra evaluada.", "Cómo alimenta la tesis final": "Establece que el valor principal del proyecto está en la construcción risk-based de cartera.", "Limitación principal": "La mejora frente al 60/40 no prueba por sí sola que crypto aporte valor."},    {"Bloque del estudio": "Capítulo 2 — Robustez y control sin crypto", "Pregunta que responde": "¿La mejora de MinVar se explica realmente por BTC/ETH o persiste sin crypto?", "Qué se implementa": "Control sin crypto, sensibilidad a lookback, crypto cap, covarianza, costes y bootstrap del delta frente al control.", "Código principal": ["src/robustness.py", "src/bootstrap.py", "src/covariance.py", "src/costs.py"], "Scripts reproducibles": ["scripts/run_robustness.py", "scripts/run_statistical_confidence.py"], "Outputs principales": ["data/processed/robustness/confidence_summary.csv", "robustness outputs", "cost sensitivity outputs"], "Resultado principal": "El delta atribuible a permitir BTC/ETH es pequeño y no robusto; el intervalo bootstrap cruza cero.", "Cómo alimenta la tesis final": "Debilita la hipótesis de crypto como diversificador estructural.", "Limitación principal": "El bootstrap ayuda a cuantificar incertidumbre, pero no elimina dependencia de muestra ni de especificación."},    {"Bloque del estudio": "Capítulo 3 — CVaR y riesgo de cola", "Pregunta que responde": "¿Optimizar pérdidas extremas cambia la conclusión sobre crypto?", "Qué se implementa": "Optimización CVaR/ES95, comparación contra MinVar y análisis de stress windows.", "Código principal": ["src/cvar_optimizer.py", "src/stress.py", "src/metrics.py"], "Scripts reproducibles": ["scripts/run_tail_risk.py"], "Outputs principales": ["data/processed/tail_risk/tail_risk_summary_net.csv", "stress summaries", "CVaR comparison outputs"], "Resultado principal": "CVaR aporta una lente de riesgo de cola, pero no domina claramente a MinVar ni rescata la tesis crypto.", "Cómo alimenta la tesis final": "Muestra que mirar pérdidas extremas no convierte BTC/ETH en asignación estructural defendible.", "Limitación principal": "La inferencia sobre cola depende de pocos episodios extremos observados."},    {"Bloque del estudio": "Capítulo 4 — Regímenes de mercado", "Pregunta que responde": "¿La lectura cambia al condicionar por estados de mercado?", "Qué se implementa": "Features de régimen, detección HMM/fallback, etiquetas de régimen, performance condicional y exposición condicionada.", "Código principal": ["src/regime_features.py", "src/regime_detection.py", "src/regime_evaluation.py"], "Scripts reproducibles": ["scripts/run_regime_analysis.py"], "Outputs principales": ["regime labels", "regime metadata", "conditional performance", "regime crypto exposure"], "Resultado principal": "Los regímenes contextualizan el riesgo: la performance cambia mucho entre riesgo contenido y estrés alto.", "Cómo alimenta la tesis final": "Apoya una lectura contextual/táctica, pero no una asignación crypto permanente.", "Limitación principal": "Los regímenes son diagnóstico histórico, no señal predictiva live validada."},    {"Bloque del estudio": "Capítulo 5 — Modelos supervisados y overlay", "Pregunta que responde": "¿Las señales supervisadas de riesgo se convierten en valor económico para la cartera?", "Qué se implementa": "Targets supervisados, features, validación temporal, modelos OOS, forecast buckets, calibración y overlay de caps dinámicos.", "Código principal": ["src/supervised_targets.py", "src/supervised_features.py", "src/supervised_validation.py", "src/supervised_models.py", "src/model_evaluation.py", "src/risk_overlay.py", "src/overlay_backtest.py"], "Scripts reproducibles": ["scripts/run_chapter5.py", "scripts/run_supervised_features.py", "scripts/run_supervised_models.py", "scripts/run_risk_overlay.py"], "Outputs principales": ["outputs/chapter5/model_scores.csv", "outputs/chapter5/model_selection.csv", "outputs/chapter5/overlay_decisions.csv", "outputs/chapter5/overlay_backtest_summary.csv"], "Resultado principal": "Existen señales predictivas en algunos targets, pero el overlay no monetiza de forma robusta frente a MinVar.", "Cómo alimenta la tesis final": "Separa capacidad predictiva de valor económico. ML/overlay añade diagnóstico, pero no cambia la conclusión sobre crypto.", "Limitación principal": "Los eventos extremos son pocos y la mejora predictiva no garantiza mejora neta de cartera."},    {"Bloque del estudio": "Síntesis final — Integración de evidencia", "Pregunta que responde": "¿Qué afirmaciones sobreviven al pipeline completo?", "Qué se implementa": "Consolidación de outputs de capítulos 1–5, comparación final de estrategias, evidencia frente a hipótesis, limitaciones y conclusión.", "Código principal": ["notebook final + helpers internos de reporting"], "Scripts reproducibles": ["no aplica como pipeline nuevo; consume outputs consolidados"], "Outputs principales": ["outputs de capítulos 1–5", "tablas finales", "figuras finales"], "Resultado principal": "MinVar sí aporta valor frente al 60/40; crypto estructural no queda respaldada; crypto aparece como exposición táctica/intermitente.", "Cómo alimenta la tesis final": "Cierra el argumento empírico completo del proyecto.", "Limitación principal": "La conclusión es válida para esta muestra, universo, datos, costes y especificación; no es una ley universal."},]def _inline_code_items(items: list[str]) -> str:    formatted = []    for item in items:        text = escape(item)        if item.endswith(".py") or item.endswith(".csv") or item.startswith("data/") or item.startswith("outputs/"):            formatted.append(f"<code>{text}</code>")        else:            formatted.append(text)    return "<br>".join(formatted)def _render_table(rows: list[dict[str, object]], columns: list[str]) -> str:    header = "".join(f"<th>{escape(column)}</th>" for column in columns)    body_rows = []    for row in rows:        cells = []        for column in columns:            value = row[column]            if isinstance(value, list):                cells.append(f"<td>{_inline_code_items(value)}</td>")            else:                cells.append(f"<td>{escape(str(value))}</td>")        body_rows.append("<tr>" + "".join(cells) + "</tr>")    return """    <style>    .chapter-map-table { border-collapse: collapse; width: 100%; table-layout: fixed; margin: 0.35rem 0 1.1rem 0; }    .chapter-map-table th { background: #d9dee3; color: #222; border: 1px solid #c8cdd2; padding: 8px; text-align: left; font-size: 12px; }    .chapter-map-table td { border: 1px solid #d9dee3; padding: 8px; vertical-align: top; font-size: 12px; line-height: 1.32; word-wrap: break-word; }    .chapter-map-table td:first-child { font-weight: 700; color: #1f4e79; width: 16%; }    .chapter-map-table code { font-family: Consolas, Menlo, monospace; background: #f2f4f5; padding: 1px 3px; border-radius: 3px; }    </style>    """ + f"<table class='chapter-map-table'><thead><tr>{header}</tr></thead><tbody>{''.join(body_rows)}</tbody></table>"conceptual_cols = ["Bloque del estudio", "Pregunta que responde", "Qué se implementa", "Resultado principal", "Cómo alimenta la tesis final"]technical_cols = ["Bloque del estudio", "Código principal", "Scripts reproducibles", "Outputs principales", "Limitación principal"]display(Markdown("### Tabla 1 — Mapa por capítulos del estudio: código, outputs y evidencia"))display(Markdown("#### Tabla 1A — Mapa conceptual por capítulos"))display(HTML(_render_table(chapter_map, conceptual_cols)))display(Markdown("#### Tabla 1B — Trazabilidad técnica por capítulos"))display(HTML(_render_table(chapter_map, technical_cols)))display(Markdown(    "**Lectura de la tabla.** La unidad de análisis es el capítulo o bloque estudiado: "    "qué pregunta responde, cómo se implementa, qué código y scripts lo reproducen, "    "qué outputs deja y cómo contribuye a la tesis final."))

### Figura 2 — MinVar frente al 60/40

**Pregunta de investigación.** ¿El framework MinVar aporta valor frente al 60/40 tradicional y cuánto de esa mejora parece venir de la construcción de cartera, no de crypto?

**Cómo se implementó**
- Código principal: `src/backtest.py`, `src/optimizer.py`, `src/benchmarks.py`
- Script: `scripts/run_backtest.py`, `scripts/run_benchmarks.py`
- Outputs usados: `data/processed/portfolio_returns.csv`, `data/processed/backtest_summary.csv`, `data/processed/benchmark_summary.csv`, `outputs/chapter5/overlay_backtest_summary.csv`
- Qué contrasta: riqueza, drawdown y métricas resumen de 60/40, MinVar y MinVar sin crypto en ventana común.
- Qué permite concluir: si la mejora principal viene de la construcción MinVar o si puede atribuirse limpiamente a BTC/ETH.

In [ ]:
if VIS_COMMON_RETURNS.empty:
    vis_note("No hay suficiente solape temporal entre 60/40 tradicional, MinVar y MinVar sin crypto para construir la Figura 2.")
    vis_show_reading(
        "La comparación en ventana común entre 60/40 tradicional, MinVar y MinVar sin crypto.",
        "No puede renderizarse porque falta al menos una de las series diarias necesarias.",
        "Sin ventana común no conviene comparar curvas finales entre estrategias con muestras distintas.",
        "La figura depende de los retornos diarios consolidados del backtest y del overlay.",
    )
else:
    summary_rows = []
    for strategy in VIS_COMMON_RETURNS.columns:
        returns = VIS_COMMON_RETURNS[strategy].dropna()
        wealth = compute_wealth(returns)
        drawdown = compute_drawdown(returns)
        ann_return = wealth.iloc[-1] ** (252 / len(returns)) - 1
        ann_vol = returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
        es95 = -returns[returns <= returns.quantile(0.05)].mean()
        summary_rows.append({
            "Estrategia": strategy,
            "Retorno anual": f"{ann_return * 100:.1f}%",
            "Sharpe": f"{sharpe:.2f}",
            "MaxDD": f"{drawdown.min() * 100:.1f}%",
            "ES95": f"{es95 * 100:.2f}%",
        })
    summary_table = pd.DataFrame(summary_rows)

    fig = plt.figure(figsize=(16, 7.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 2, height_ratios=[1.1, 0.56], width_ratios=[1.1, 1.0])
    ax_wealth = fig.add_subplot(grid[0, 0])
    ax_dd = fig.add_subplot(grid[0, 1])
    ax_table = fig.add_subplot(grid[1, :])

    for strategy in VIS_COMMON_WEALTH.columns:
        color = VIS_COLORS.get(strategy, None)
        ax_wealth.plot(VIS_COMMON_WEALTH.index, VIS_COMMON_WEALTH[strategy], linewidth=2.2, color=color, label=strategy)
        ax_dd.plot(VIS_COMMON_DRAWDOWN.index, VIS_COMMON_DRAWDOWN[strategy] * 100.0, linewidth=2.0, color=color, label=strategy)

    ax_wealth.set_title("Panel A. Riqueza acumulada", fontsize=12)
    ax_wealth.set_xlabel("Fecha")
    ax_wealth.set_ylabel("Índice de riqueza (base = 1)")
    ax_wealth.grid(True, alpha=0.25)

    ax_dd.set_title("Panel B. Drawdowns", fontsize=12)
    ax_dd.set_xlabel("Fecha")
    ax_dd.set_ylabel("Drawdown (%)")
    ax_dd.grid(True, alpha=0.25)

    handles, labels = ax_wealth.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=3, frameon=False, title="Estrategia")

    ax_table.axis("off")
    table = ax_table.table(cellText=summary_table.values, colLabels=summary_table.columns, cellLoc="center", colLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.45)
    for row_idx in range(len(summary_table) + 1):
        for col_idx in range(len(summary_table.columns)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor("#dddddd")
            if row_idx == 0:
                cell.set_facecolor("#eaf2f8")
                cell.set_text_props(weight="bold")
    ax_table.set_title("Panel C. Métricas en la misma ventana", fontsize=12, pad=8)

    fig.suptitle("Figura 2. MinVar frente al 60/40", fontsize=15)
    plt.show()

    vis_show_reading(
        f"Riqueza, drawdown y métricas resumen de 60/40, MinVar y MinVar sin crypto sobre la ventana común {VIS_COMMON_RETURNS.index.min().date()} a {VIS_COMMON_RETURNS.index.max().date()}.",
        "MinVar mejora el perfil frente al 60/40, pero MinVar sin crypto queda muy cerca de MinVar en las métricas centrales.",
        "La figura separa el valor de la construcción MinVar del valor incremental de crypto: si el control sin crypto queda cerca, la mejora no puede atribuirse limpiamente a BTC/ETH.",
        "Las métricas del panel C se recalculan sobre la ventana común para hacer comparables las curvas; pueden diferir ligeramente de tablas con ventanas nativas.",
    )

### Figura 3 — Efecto incremental de crypto

**Pregunta de investigación.** ¿Permitir BTC/ETH añade una mejora material y robusta frente al mismo MinVar sin crypto?

**Cómo se implementó**
- Código principal: `src/robustness.py`, `src/bootstrap.py`
- Script: `scripts/run_robustness.py`, `scripts/run_statistical_confidence.py`
- Outputs usados: `outputs/chapter5/overlay_backtest_summary.csv`, `data/processed/robustness/confidence_summary.csv`
- Qué contrasta: deltas MinVar menos MinVar sin crypto, intervalo bootstrap y exposición efectiva.
- Qué permite concluir: si crypto aporta algo positivo, material, robusto y estructural.

**Nota de lectura.** El bootstrap sirve para comprobar si el delta observado parece estable o podría deberse al ruido de la muestra. No es una prueba definitiva. La secuencia de interpretación es estricta: positivo no implica material; material no implica robusto; robusto no implica estructural.

In [ ]:
if VIS_BASELINE_ROW.empty or VIS_NO_CRYPTO_ROW.empty:
    vis_note("No se encuentran simultáneamente MinVar y MinVar sin crypto en `overlay_backtest_summary.csv`.")
    vis_show_reading(
        "Los deltas de rendimiento, riesgo y exposición entre MinVar y el control sin crypto.",
        "La figura no puede cuantificarse porque falta al menos una de las filas resumen necesarias.",
        "Sin el control explícito la atribución del efecto crypto queda incompleta.",
        "La comparación depende de la tabla resumen neta del capítulo de overlay.",
    )
else:
    delta_return = (vis_metric(VIS_BASELINE_ROW, "ann_return", "annualized_return") - vis_metric(VIS_NO_CRYPTO_ROW, "ann_return", "annualized_return")) * 100.0
    delta_maxdd = (vis_metric(VIS_BASELINE_ROW, "max_drawdown") - vis_metric(VIS_NO_CRYPTO_ROW, "max_drawdown")) * 100.0
    delta_es95 = (vis_metric(VIS_BASELINE_ROW, "es95", "expected_shortfall_net") - vis_metric(VIS_NO_CRYPTO_ROW, "es95", "expected_shortfall_net")) * 100.0
    delta_sharpe = vis_metric(VIS_BASELINE_ROW, "sharpe") - vis_metric(VIS_NO_CRYPTO_ROW, "sharpe")
    avg_crypto = vis_metric(VIS_BASELINE_ROW, "avg_crypto_weight", "crypto_avg_weight")
    max_crypto = vis_metric(VIS_BASELINE_ROW, "max_crypto_weight", "crypto_max_weight")
    n_gt2 = vis_metric(VIS_BASELINE_ROW, "n_rebalances_crypto_gt_2pct", "n_crypto_rebalances_gt_2pct")
    n_rebalances = len(VIS_CRYPTO_PANEL.loc[VIS_CRYPTO_PANEL["Estrategia"] == "MinVar"]) if not VIS_CRYPTO_PANEL.empty else np.nan
    share_gt2 = n_gt2 / n_rebalances if pd.notna(n_gt2) and pd.notna(n_rebalances) and n_rebalances else np.nan

    anchor = pd.DataFrame()
    if VIS_CONFIDENCE is not None and not VIS_CONFIDENCE.empty and "comparison_id" in VIS_CONFIDENCE.columns:
        anchor = VIS_CONFIDENCE.loc[VIS_CONFIDENCE["comparison_id"] == "C1_anchor_pair"]

    fig = plt.figure(figsize=(15.5, 7.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 3, height_ratios=[1.05, 0.85], width_ratios=[1.0, 1.0, 1.0])
    ax_sharpe = fig.add_subplot(grid[0, 0])
    ax_return = fig.add_subplot(grid[0, 1])
    ax_risk = fig.add_subplot(grid[0, 2])
    ax_expo = fig.add_subplot(grid[1, :])

    ax_sharpe.axhspan(-0.05, 0.05, color="#f2f2f2", alpha=0.9, label="banda débil ±0.05")
    ax_sharpe.axhline(0, color=COLOR["zero"], linewidth=1.2)
    ax_sharpe.axhline(0.05, color=COLOR["threshold"], linestyle="--", linewidth=0.9)
    ax_sharpe.axhline(-0.05, color=COLOR["threshold"], linestyle="--", linewidth=0.9)
    ax_sharpe.set_ylim(-0.08, 0.08)
    ax_sharpe.set_xlim(-0.6, 0.6)
    ax_sharpe.set_xticks([0], ["Δ Sharpe"])
    ax_sharpe.set_title("Panel A. Zoom de Δ Sharpe", fontsize=12)
    ax_sharpe.set_ylabel("Sharpe")
    ax_sharpe.grid(True, axis="y", alpha=0.25)
    if not anchor.empty:
        row = anchor.iloc[0]
        point = float(row["point_estimate_difference"])
        lower = float(row["ci_lower"])
        upper = float(row["ci_upper"])
        ax_sharpe.errorbar([0], [point], yerr=[[point - lower], [upper - point]], fmt="o", color=COLOR["minvar"], capsize=5, linewidth=1.6)
        ax_sharpe.text(0.0, min(0.075, upper + 0.004), "IC bootstrap", ha="center", va="bottom", fontsize=9)
    else:
        ax_sharpe.bar([0], [delta_sharpe], color=COLOR["minvar"])
    ax_sharpe.text(0, delta_sharpe, f"{delta_sharpe:+.3f}", ha="center", va="bottom" if delta_sharpe >= 0 else "top", fontsize=9)

    ax_return.axhspan(-1.0, 1.0, color="#f2f2f2", alpha=0.9)
    ax_return.axhline(0, color=COLOR["zero"], linewidth=1.2)
    ax_return.axhline(1.0, color=COLOR["threshold"], linestyle="--", linewidth=0.9)
    ax_return.axhline(-1.0, color=COLOR["threshold"], linestyle="--", linewidth=0.9)
    ax_return.bar(["Δ retorno"], [delta_return], color=COLOR["minvar"] if delta_return >= 0 else "#8c6d31")
    ax_return.set_ylim(-1.5, 1.5)
    ax_return.set_title("Panel B. Zoom de Δ retorno", fontsize=12)
    ax_return.set_ylabel("Delta (p.p.)")
    ax_return.grid(True, axis="y", alpha=0.25)
    ax_return.text(0, delta_return, f"{delta_return:+.2f}", ha="center", va="bottom" if delta_return >= 0 else "top", fontsize=9)

    risk_df = pd.DataFrame({"Métrica": ["Δ MaxDD", "Δ ES95"], "Delta": [delta_maxdd, delta_es95]})
    bars = ax_risk.bar(risk_df["Métrica"], risk_df["Delta"], color=["#1f4e79" if v >= 0 else "#8c6d31" for v in risk_df["Delta"]])
    ax_risk.axhline(0, color=COLOR["zero"], linewidth=1.2)
    ax_risk.set_title("Panel C. Riesgo incremental", fontsize=12)
    ax_risk.set_ylabel("Delta (p.p.)")
    ax_risk.grid(True, axis="y", alpha=0.25)
    for bar, value in zip(bars, risk_df["Delta"]):
        ax_risk.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:+.2f}", ha="center", va="bottom" if value >= 0 else "top", fontsize=9)

    expo_table = pd.DataFrame([
        ["Peso crypto medio", f"{avg_crypto * 100:.2f}%"],
        ["Peso crypto máximo", f"{max_crypto * 100:.2f}%"],
        ["Rebalances crypto >2%", f"{int(n_gt2) if pd.notna(n_gt2) else 'n/d'}"],
        ["Frecuencia >2%", f"{share_gt2 * 100:.1f}%" if pd.notna(share_gt2) else "n/d"],
        ["Lectura", "positivo ≠ material ≠ robusto ≠ estructural"],
    ], columns=["Indicador", "Valor"])
    ax_expo.axis("off")
    table = ax_expo.table(cellText=expo_table.values, colLabels=expo_table.columns, cellLoc="left", colLoc="left", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10.2)
    table.scale(1, 1.45)
    for row_idx in range(len(expo_table) + 1):
        for col_idx in range(len(expo_table.columns)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor("#dddddd")
            if row_idx == 0:
                cell.set_facecolor(COLOR["neutral"])
                cell.set_text_props(weight="bold")
    ax_expo.set_title("Panel D. Exposición efectiva y regla de interpretación", fontsize=12, pad=8)

    fig.suptitle("Figura 3. Efecto incremental de crypto", fontsize=15)
    plt.show()

    obs_text = "El delta de Sharpe es pequeño y el intervalo bootstrap incluye cero." if not anchor.empty and bool(anchor.iloc[0]["ci_includes_zero"]) else "Las diferencias son pequeñas incluso cuando el signo favorece a la versión con crypto."
    vis_show_reading(
        "Deltas de Sharpe y retorno con zoom alrededor de cero, deltas de riesgo y exposición efectiva.",
        obs_text + f" Δ retorno = {delta_return:+.2f} p.p.; Δ Sharpe = {delta_sharpe:+.3f}; peso medio crypto = {avg_crypto * 100:.2f}%.",
        "El bootstrap es una forma humana de preguntar si el delta observado parece estable o si puede ser ruido de muestra: positivo no significa material; material no significa robusto; robusto no significa estructural.",
        "No demuestra que BTC/ETH aporten diversificación estructural: para eso harían falta materialidad, robustez y exposición persistente.",
    )

### Figura 4 — Exposición crypto: estructural o táctica

**Pregunta de investigación.** ¿La asignación a BTC/ETH aparece como exposición persistente y material, o como uso intermitente de baja intensidad?

**Cómo se implementó**
- Código principal: `src/backtest.py`, `src/optimizer.py`
- Script: `scripts/run_backtest.py`
- Outputs usados: `data/processed/weights_history.csv`, `outputs/chapter5/overlay_backtest_summary.csv`
- Qué contrasta: peso total BTC+ETH de MinVar por rebalanceo, distribución por buckets y estadísticos de exposición.
- Qué permite concluir: si crypto se comporta como asignación estructural o como exposición táctica.

In [ ]:
minvar_crypto = pd.DataFrame()
if VIS_CRYPTO_PANEL is not None and not VIS_CRYPTO_PANEL.empty:
    minvar_crypto = VIS_CRYPTO_PANEL.loc[VIS_CRYPTO_PANEL["Estrategia"] == "MinVar"].sort_values("rebalance_date").copy()

if minvar_crypto.empty:
    vis_note("No hay panel consolidado de pesos MinVar para construir la Figura 4.")
    vis_show_reading(
        "La evolución temporal, distribución y resumen estadístico de la exposición total a BTC y ETH.",
        "La figura no puede renderizarse porque faltan historiales de pesos MinVar.",
        "Sin pesos por rebalanceo no puede evaluarse si crypto es estructural o solo táctica.",
        "La visual depende de `weights_history` y de la consolidación de pesos crypto.",
    )
else:
    focus = minvar_crypto["crypto_total"].astype(float)
    bucket_df = minvar_crypto.copy()
    bucket_df["Tramo"] = pd.cut(
        bucket_df["crypto_total"],
        bins=[-1e-12, 1e-8, 0.01, 0.02, 1.0],
        labels=["0%", "0-1%", "1-2%", ">2%"],
        include_lowest=True,
    )
    bucket_share = bucket_df["Tramo"].value_counts(normalize=True).reindex(["0%", "0-1%", "1-2%", ">2%"], fill_value=0.0)
    exposure_stats = pd.DataFrame([
        ["Media", f"{focus.mean() * 100:.2f}%"],
        ["Mediana", f"{focus.median() * 100:.2f}%"],
        ["Máximo", f"{focus.max() * 100:.2f}%"],
        ["Rebalances >2%", f"{(focus > 0.02).sum()} / {len(focus)}"],
        ["Frecuencia >2%", f"{(focus > 0.02).mean() * 100:.1f}%"],
    ], columns=["Estadístico", "Valor"])

    fig = plt.figure(figsize=(16, 6.4), constrained_layout=True)
    grid = fig.add_gridspec(1, 3, width_ratios=[1.55, 0.95, 0.95])
    ax_time = fig.add_subplot(grid[0, 0])
    ax_bucket = fig.add_subplot(grid[0, 1])
    ax_table = fig.add_subplot(grid[0, 2])

    ax_time.plot(minvar_crypto["rebalance_date"], minvar_crypto["crypto_total"] * 100.0, linewidth=2.2, color=VIS_COLORS.get("MinVar"), label="MinVar")
    ax_time.axhline(1.0, color="#777777", linestyle=":", linewidth=1.0)
    ax_time.axhline(2.0, color="#444444", linestyle="--", linewidth=1.1)
    ax_time.set_title("Panel A. Peso total BTC+ETH", fontsize=12)
    ax_time.set_xlabel("Fecha de rebalanceo")
    ax_time.set_ylabel("Peso crypto (%)")
    ax_time.grid(True, alpha=0.25)
    ax_time.legend(frameon=False)

    bucket_colors = ["#d9d9d9", "#9ecae1", "#fdd0a2", "#fb6a4a"]
    bars = ax_bucket.bar(bucket_share.index.astype(str), bucket_share.values * 100.0, color=bucket_colors)
    ax_bucket.set_title("Panel B. Distribución por buckets", fontsize=12)
    ax_bucket.set_ylabel("Rebalanceos (%)")
    ax_bucket.grid(True, axis="y", alpha=0.25)
    for bar, value in zip(bars, bucket_share.values):
        ax_bucket.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value * 100:.1f}%", ha="center", va="bottom", fontsize=9)

    ax_table.axis("off")
    table = ax_table.table(cellText=exposure_stats.values, colLabels=exposure_stats.columns, cellLoc="left", colLoc="left", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10.5)
    table.scale(1.0, 1.55)
    for row_idx in range(len(exposure_stats) + 1):
        for col_idx in range(len(exposure_stats.columns)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor("#dddddd")
            if row_idx == 0:
                cell.set_facecolor("#e9eef3")
                cell.set_text_props(weight="bold")
    ax_table.set_title("Panel C. Resumen de exposición", fontsize=12, pad=8)

    fig.suptitle("Figura 4. Exposición crypto: estructural o táctica", fontsize=15)
    plt.show()

    vis_show_reading(
        "La senda temporal del peso total BTC+ETH de MinVar, su distribución por buckets y un resumen estadístico compacto.",
        f"La exposición media es {focus.mean() * 100:.2f}%, la mediana es {focus.median() * 100:.2f}% y solo supera 2% en {(focus > 0.02).mean() * 100:.1f}% de rebalanceos.",
        "Una asignación estructural debería aparecer de forma frecuente y material; la evidencia muestra pesos bajos e intermitentes.",
        "No demuestra que crypto sea inútil en cualquier contexto; sí limita la tesis de asignación estructural dentro de este diseño.",
    )

### Figura 5 — Robustez de la tesis crypto

**Pregunta de investigación.** ¿La conclusión sobre crypto depende de un supuesto concreto, o se mantiene al cambiar ventanas, costes, topes y especificaciones?

**Cómo se implementó**
- Código principal: `src/robustness.py`, `src/bootstrap.py`
- Script: `scripts/run_robustness.py`, `scripts/run_statistical_confidence.py`
- Outputs usados: `data/processed/robustness/robustness_summary_common_family_net.csv`, `data/processed/robustness/confidence_summary.csv`
- Qué contrasta: MinVar vs MinVar sin crypto por familia de robustez, deltas netos y sensibilidad a costes.
- Qué permite concluir: si el claim crypto sobrevive a cambios razonables de supuestos.

In [ ]:
if VIS_ROBUSTNESS is None or VIS_ROBUSTNESS.empty:
    vis_note("No está disponible `robustness_summary_common_family_net.csv`.")
    vis_show_reading(
        "La comparación MinVar vs MinVar sin crypto por familias de robustez.",
        "La figura no puede construirse porque falta la tabla consolidada de robustez en ventana común.",
        "Sin esta capa no puede evaluarse cuán estable es la mejora atribuida a crypto.",
        "La lectura depende del resumen neto por familias del bloque de robustez.",
    )
else:
    rob = VIS_ROBUSTNESS.copy()
    rob["Estrategia"] = np.where(rob["max_total_crypto_weight"].astype(float).gt(0), "MinVar", "MinVar sin crypto")
    family_labels = {"lookback": "Ventana de estimación", "crypto_cap": "Límite máximo crypto", "covariance_method": "Método de covarianza", "rebalance_frequency": "Frecuencia de rebalanceo", "cost_sensitivity": "Costes de transacción"}
    family_order = [fam for fam in ["lookback", "crypto_cap", "covariance_method", "rebalance_frequency", "cost_sensitivity"] if fam in rob["family"].unique()]
    if not family_order:
        family_order = sorted(rob["family"].dropna().unique())

    family_summary = rob.loc[rob["Estrategia"].isin(["MinVar", "MinVar sin crypto"])].groupby(["family", "Estrategia"], as_index=False).agg(
        sharpe_net=("sharpe_net", "mean"),
        ann_return_net=("ann_return_net", "mean"),
        max_drawdown_net=("max_drawdown_net", "mean"),
    )
    sharpe_pivot = family_summary.pivot(index="family", columns="Estrategia", values="sharpe_net").reindex(family_order)
    return_pivot = family_summary.pivot(index="family", columns="Estrategia", values="ann_return_net").reindex(family_order)
    dd_pivot = family_summary.pivot(index="family", columns="Estrategia", values="max_drawdown_net").reindex(family_order)

    delta_sharpe = (sharpe_pivot.get("MinVar") - sharpe_pivot.get("MinVar sin crypto")).dropna()
    delta_return = ((return_pivot.get("MinVar") - return_pivot.get("MinVar sin crypto")) * 100.0).dropna()
    delta_maxdd = ((dd_pivot.get("MinVar") - dd_pivot.get("MinVar sin crypto")) * 100.0).dropna()

    display_order = [family_labels.get(fam, str(fam).replace("_", " ").title()) for fam in family_order]

    fig, axes = plt.subplots(2, 2, figsize=(15.8, 8.4))

    axes[0, 0].bar(display_order[:len(delta_sharpe)], delta_sharpe.values, color=[COLOR["positive"] if v >= 0 else COLOR["negative"] for v in delta_sharpe.values])
    axes[0, 0].axhline(0, color=COLOR["zero"], linewidth=1.1)
    axes[0, 0].axhspan(-0.05, 0.05, color="#f2f2f2", alpha=0.85)
    axes[0, 0].set_title("Panel A. Δ Sharpe por familia", fontsize=12)
    axes[0, 0].set_ylabel("MinVar - control")
    axes[0, 0].tick_params(axis="x", rotation=20)
    axes[0, 0].grid(True, axis="y", alpha=0.25)

    axes[0, 1].bar(display_order[:len(delta_return)], delta_return.values, color=[COLOR["positive"] if v >= 0 else COLOR["negative"] for v in delta_return.values])
    axes[0, 1].axhline(0, color=COLOR["zero"], linewidth=1.1)
    axes[0, 1].axhspan(-1.0, 1.0, color="#f2f2f2", alpha=0.85)
    axes[0, 1].set_title("Panel B. Δ retorno anual por familia", fontsize=12)
    axes[0, 1].set_ylabel("Delta (p.p.)")
    axes[0, 1].tick_params(axis="x", rotation=20)
    axes[0, 1].grid(True, axis="y", alpha=0.25)

    axes[1, 0].bar(display_order[:len(delta_maxdd)], delta_maxdd.values, color=[COLOR["negative"] if v > 0 else COLOR["positive"] for v in delta_maxdd.values])
    axes[1, 0].axhline(0, color=COLOR["zero"], linewidth=1.1)
    axes[1, 0].set_title("Panel C. Δ MaxDD por familia", fontsize=12)
    axes[1, 0].set_ylabel("Delta (p.p.)")
    axes[1, 0].tick_params(axis="x", rotation=20)
    axes[1, 0].grid(True, axis="y", alpha=0.25)

    axes[1, 1].axis("off")
    robustness_summary = pd.DataFrame([
        ["Familias comparables", len(delta_sharpe)],
        ["Mayor |Δ Sharpe|", f"{delta_sharpe.abs().max():.3f}" if not delta_sharpe.empty else "n/d"],
        ["Mayor |Δ retorno|", f"{delta_return.abs().max():.2f} p.p." if not delta_return.empty else "n/d"],
        ["Mayor |Δ MaxDD|", f"{delta_maxdd.abs().max():.2f} p.p." if not delta_maxdd.empty else "n/d"],
        ["Lectura", "deltas pequeños frente a materialidad"],
    ], columns=["Indicador", "Valor"])
    table = axes[1, 1].table(cellText=robustness_summary.values, colLabels=robustness_summary.columns, cellLoc="left", colLoc="left", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10.2)
    table.scale(1.0, 1.45)
    for row_idx in range(len(robustness_summary) + 1):
        for col_idx in range(len(robustness_summary.columns)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor("#dddddd")
            if row_idx == 0:
                cell.set_facecolor(COLOR["neutral"])
                cell.set_text_props(weight="bold")
    axes[1, 1].set_title("Panel D. Resumen de robustez", fontsize=12, pad=8)

    fig.suptitle("Figura 5. Robustez de la tesis crypto", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

    max_abs_delta = delta_sharpe.abs().max() if not delta_sharpe.empty else np.nan
    vis_show_reading(
        "Deltas frente al control sin crypto por familias comparables: ventana, límite máximo crypto, método de covarianza, frecuencia y costes.",
        f"El mayor |Δ Sharpe| por familia es {max_abs_delta:.3f}; las bandas grises marcan diferencias débiles frente a referencias de materialidad." if pd.notna(max_abs_delta) else "No hay deltas comparables suficientes por familia.",
        "La pregunta no es si alguna barra tiene signo positivo, sino si el delta sobrevive a supuestos razonables con magnitud económica.",
        "No demuestra robustez estructural de crypto; una mejora futura razonable sería reportar también intervalos por familia si se generan en Chapter 2.",
    )

### Figura 6 — CVaR y riesgo de cola

**Pregunta de investigación.** ¿Optimizar pérdidas extremas mejora de forma clara ES95 y MaxDD frente a MinVar y cambia la tesis sobre crypto?

**Cómo se implementó**
- Código principal: `src/cvar_optimizer.py`, `src/stress.py`
- Script: `scripts/run_tail_risk.py`
- Outputs usados: `data/processed/tail_risk/tail_risk_summary_net.csv`, `data/processed/tail_risk/stress_summary.csv`
- Qué contrasta: MinVar, MinVar sin crypto, CVaR y CVaR sin crypto en retorno, ES95, drawdown y deltas CVaR - MinVar.
- Qué permite concluir: si el objetivo de cola domina económicamente o solo aporta un ángulo diagnóstico.

**Nota de lectura.** CVaR es un test de riesgo de cola, no una estrategia ganadora por defecto. Si no reduce de forma clara ES95 o MaxDD sin sacrificar otras métricas, no rescata la tesis crypto.

In [ ]:
if VIS_TAIL_NET is None or VIS_TAIL_NET.empty:
	vis_note("No está disponible `tail_risk_summary_net.csv`.")
	vis_show_reading(
		"La comparación entre MinVar y CVaR en retorno, cola y drawdown.",
		"La figura no puede renderizarse porque falta el resumen neto de tail risk.",
		"Sin esta tabla no puede evaluarse si CVaR domina o no a MinVar.",
		"La visual depende del resumen neto y de `stress_summary.csv` para estrés.",
	)
else:
	tail = VIS_TAIL_NET.copy()
	tail["Estrategia"] = tail["strategy"].map(vis_h_strategy)
	ordered = ["MinVar", "MinVar sin crypto", "CVaR", "CVaR sin crypto"]
	tail = tail.loc[tail["Estrategia"].isin(ordered)].copy()
	tail["order"] = tail["Estrategia"].map({name: idx for idx, name in enumerate(ordered)})
	tail = tail.sort_values("order")

	cvar_row = vis_pick_row(VIS_TAIL_NET, "strategy", "cvar_baseline")
	minvar_row = vis_pick_row(VIS_TAIL_NET, "strategy", "minvar_baseline_ch1")
	deltas = []
	if not cvar_row.empty and not minvar_row.empty:
		deltas = [
			("Delta Sharpe", vis_metric(cvar_row, "sharpe_net") - vis_metric(minvar_row, "sharpe_net"), "más alto es mejor"),
			("Delta ES95", (vis_metric(cvar_row, "expected_shortfall_net") - vis_metric(minvar_row, "expected_shortfall_net")) * 100.0, "negativo reduce pérdida"),
			("Delta MaxDD", (abs(vis_metric(cvar_row, "max_drawdown_net")) - abs(vis_metric(minvar_row, "max_drawdown_net"))) * 100.0, "negativo reduce pérdida"),
		]

	fig = plt.figure(figsize=(16.5, 7.4), constrained_layout=True)
	grid = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.86], width_ratios=[1.25, 1.05, 1.05])
	ax_scatter = fig.add_subplot(grid[:, 0])
	ax_table = fig.add_subplot(grid[:, 1])
	ax_cards = fig.add_subplot(grid[:, 2])

	for row in tail.itertuples():
		ax_scatter.scatter(row.ann_return_net * 100.0, row.expected_shortfall_net * 100.0, s=105, color=VIS_COLORS.get(row.Estrategia), label=row.Estrategia)
		ax_scatter.annotate(row.Estrategia, (row.ann_return_net * 100.0, row.expected_shortfall_net * 100.0), textcoords="offset points", xytext=(6, 6), fontsize=9)
	ax_scatter.set_title("Panel A. Retorno anual vs ES95", fontsize=12)
	ax_scatter.set_xlabel("Retorno anual (%)")
	ax_scatter.set_ylabel("ES95 (% pérdida; menor es mejor)")
	ax_scatter.grid(True, alpha=0.25, color=COLOR["grid"])

	compact = tail[["Estrategia", "sharpe_net", "expected_shortfall_net", "max_drawdown_net"]].copy()
	compact["Sharpe"] = compact["sharpe_net"].map(lambda x: f"{x:.3f}")
	compact["ES95"] = compact["expected_shortfall_net"].map(lambda x: f"{x * 100:.2f}%")
	compact["MaxDD"] = compact["max_drawdown_net"].map(lambda x: f"{abs(x) * 100:.1f}%")
	compact_view = compact[["Estrategia", "Sharpe", "ES95", "MaxDD"]]
	ax_table.axis("off")
	table = ax_table.table(cellText=compact_view.values, colLabels=compact_view.columns, cellLoc="left", colLoc="left", loc="center")
	table.auto_set_font_size(False)
	table.set_fontsize(9.5)
	table.scale(1.0, 1.48)
	for row_idx in range(len(compact_view) + 1):
		for col_idx in range(len(compact_view.columns)):
			cell = table[row_idx, col_idx]
			cell.set_edgecolor("#dddddd")
			if row_idx == 0:
				cell.set_facecolor(COLOR["neutral"])
				cell.set_text_props(weight="bold")
	ax_table.set_title("Panel B. Riesgo de cola y drawdown", fontsize=12, pad=8)

	ax_cards.axis("off")
	ax_cards.set_title("Panel C. Deltas CVaR - MinVar", fontsize=12, pad=8)
	if not deltas:
		ax_cards.text(0.5, 0.5, "Sin deltas CVaR-MinVar", ha="center", va="center", fontsize=11)
	else:
		y_positions = [0.72, 0.46, 0.20]
		for (label, value, note), y in zip(deltas, y_positions):
			better = value > 0 if label == "Delta Sharpe" else value < 0
			face = COLOR["risk_contained_light"] if better else COLOR["high_stress_light"]
			edge = COLOR["positive"] if better else COLOR["negative"]
			box = FancyBboxPatch((0.08, y), 0.84, 0.18, boxstyle="round,pad=0.018,rounding_size=0.02", linewidth=1.1, edgecolor=edge, facecolor=face)
			ax_cards.add_patch(box)
			value_text = f"{value:+.3f}" if label == "Delta Sharpe" else f"{value:+.2f} p.p."
			ax_cards.text(0.13, y + 0.118, label, ha="left", va="center", fontsize=10, weight="bold")
			ax_cards.text(0.87, y + 0.118, value_text, ha="right", va="center", fontsize=12, weight="bold")
			ax_cards.text(0.13, y + 0.045, note, ha="left", va="center", fontsize=8.7, color="#555555")
	ax_cards.set_xlim(0, 1)
	ax_cards.set_ylim(0, 1)

	fig.suptitle("Figura 6. CVaR y riesgo de cola", fontsize=15)
	plt.show()

	if not cvar_row.empty and not minvar_row.empty:
		obs_text = (
			f"CVaR entrega Sharpe neto {vis_metric(cvar_row, 'sharpe_net'):.3f} frente a {vis_metric(minvar_row, 'sharpe_net'):.3f} en MinVar, "
			f"con ES95 de {vis_metric(cvar_row, 'expected_shortfall_net') * 100.0:.2f}% frente a {vis_metric(minvar_row, 'expected_shortfall_net') * 100.0:.2f}%."
		)
	else:
		obs_text = "La comparación directa CVaR-MinVar no está completa."
	vis_show_reading(
		"Retorno frente a ES95, tabla compacta de Sharpe/ES95/MaxDD y deltas separados para evitar mezclar unidades en un único eje.",
		obs_text,
		"CVaR es una prueba de riesgo de cola. Si no mejora claramente ES95 o MaxDD frente a MinVar, no rescata la tesis estructural crypto.",
		"ES95 se muestra como pérdida positiva: menor es mejor. MaxDD se lee como magnitud de pérdida: menor magnitud es mejor.",
	)

### Figura 7 — Regímenes de mercado

**Pregunta de investigación.** ¿El contexto histórico de mercado cambia la interpretación de resultados sin convertirse en una señal predictiva?

**Cómo se implementó**
- Código principal: `src/regime_features.py`, `src/regime_detection.py`, `src/regime_evaluation.py`
- Script: `scripts/run_regime_analysis.py`
- Outputs usados: `data/processed/regime_analysis/regime_labels.csv`, `data/processed/regime_analysis/regime_features.csv`, `data/processed/regime_analysis/regime_conditional_performance_net.csv`, `data/processed/regime_analysis/regime_crypto_exposure.csv`
- Qué contrasta: indicador observable de estrés, bandas de régimen, duración de estados, performance condicional y exposición crypto.
- Qué permite concluir: si los resultados son dependientes del entorno y si crypto sigue siendo de baja intensidad.

In [ ]:
regime_features = safe_read_csv(REGIME_DIR / "regime_features.csv", parse_dates=["date"])
regime_palette = {"Riesgo contenido": COLOR["risk_contained"], "Estrés alto": COLOR["high_stress"]}
regime_light = {"Riesgo contenido": COLOR["risk_contained_light"], "Estrés alto": COLOR["high_stress_light"]}


def _episode_stats(timeline: pd.DataFrame) -> pd.DataFrame:
	if timeline.empty:
		return pd.DataFrame(columns=["Régimen", "% observaciones", "Episodios", "Duración media", "Duración máxima"])
	ordered = timeline.sort_values("date").copy()
	ordered["episode"] = ordered["Régimen"].ne(ordered["Régimen"].shift()).cumsum()
	spans = ordered.groupby(["Régimen", "episode"]).agg(start=("date", "min"), end=("date", "max"), days=("date", "size")).reset_index()
	counts = ordered["Régimen"].value_counts(normalize=True)
	rows = []
	for regime_name in ["Riesgo contenido", "Estrés alto"]:
		sub = spans.loc[spans["Régimen"] == regime_name]
		rows.append({
			"Régimen": regime_name,
			"% observaciones": f"{counts.get(regime_name, 0.0) * 100:.1f}%",
			"Episodios": int(len(sub)),
			"Duración media": f"{sub['days'].mean():.0f} días" if not sub.empty else "0 días",
			"Duración máxima": f"{sub['days'].max():.0f} días" if not sub.empty else "0 días",
		})
	return pd.DataFrame(rows)


if (VIS_REGIME_LABELS is None or VIS_REGIME_LABELS.empty) and (VIS_REGIME_PERF is None or VIS_REGIME_PERF.empty):
	vis_note("Faltan `regime_labels.csv` y `regime_conditional_performance_net.csv`.")
	vis_show_reading(
		"La cronología de regímenes y su relación con performance y exposición crypto.",
		"La figura no puede renderizarse porque faltan los outputs básicos del análisis de regímenes.",
		"Sin esta capa solo queda la comparación agregada, sin contexto de entorno.",
		"Los regímenes se tratan aquí como diagnóstico histórico, no como señal operativa.",
	)
else:
	fig = plt.figure(figsize=(16.5, 9.0), constrained_layout=True)
	grid = fig.add_gridspec(2, 2, height_ratios=[1.15, 1.0])
	ax_indicator = fig.add_subplot(grid[0, :])
	ax_duration = fig.add_subplot(grid[1, 0])
	ax_perf = fig.add_subplot(grid[1, 1])

	regime_view = pd.DataFrame()
	if VIS_REGIME_LABELS is None or VIS_REGIME_LABELS.empty or regime_features is None or regime_features.empty:
		ax_indicator.text(0.5, 0.5, "Sin indicador observable de régimen", ha="center", va="center", fontsize=11)
		ax_indicator.axis("off")
	else:
		timeline = VIS_REGIME_LABELS.copy().sort_values("date")
		timeline["Régimen"] = timeline["regime_name"].map(vis_h_regime)
		feat = regime_features.copy().sort_values("date")
		indicator_col = "realized_vol_spy_63d" if "realized_vol_spy_63d" in feat.columns else "drawdown_spy_126d"
		regime_view = timeline.merge(feat[["date", indicator_col]], on="date", how="left")
		regime_view["Indicador"] = regime_view[indicator_col].abs() * 100.0
		regime_view["episode"] = regime_view["Régimen"].ne(regime_view["Régimen"].shift()).cumsum()

		for _, span in regime_view.groupby("episode"):
			regime_name = span["Régimen"].iloc[0]
			ax_indicator.axvspan(span["date"].min(), span["date"].max(), color=regime_light.get(regime_name, COLOR["neutral"]), alpha=0.65, linewidth=0)
		ax_indicator.plot(regime_view["date"], regime_view["Indicador"], color=COLOR["threshold"], linewidth=1.8, label="Volatilidad SPY 63d" if indicator_col == "realized_vol_spy_63d" else "Drawdown SPY 126d")
		ax_indicator.set_title("Panel A. Indicador observable con bandas de régimen", fontsize=12)
		ax_indicator.set_ylabel("Volatilidad SPY 63d (%)" if indicator_col == "realized_vol_spy_63d" else "Magnitud drawdown SPY 126d (%)")
		ax_indicator.set_xlabel("Fecha")
		ax_indicator.grid(True, alpha=0.25, color=COLOR["grid"])
		for label, date_hint in [("COVID", "2020-03-15"), ("Subidas de tipos", "2022-06-15")]:
			date = pd.Timestamp(date_hint)
			if regime_view["date"].min() <= date <= regime_view["date"].max():
				yval = regime_view.loc[(regime_view["date"] - date).abs().idxmin(), "Indicador"]
				ax_indicator.annotate(label, xy=(date, yval), xytext=(6, 12), textcoords="offset points", fontsize=8.5, color="#555555", arrowprops={"arrowstyle": "-", "color": COLOR["threshold"], "lw": 0.8})
		handles = [plt.Line2D([0], [0], color=regime_palette[name], lw=6, alpha=0.8, label=name) for name in ["Riesgo contenido", "Estrés alto"]]
		handles.append(plt.Line2D([0], [0], color=COLOR["threshold"], lw=1.8, label="Indicador observable"))
		ax_indicator.legend(handles=handles, frameon=False, loc="upper left")

	duration_table = _episode_stats(regime_view)
	if duration_table.empty:
		ax_duration.text(0.5, 0.5, "Sin estadística de episodios", ha="center", va="center", fontsize=11)
		ax_duration.axis("off")
	else:
		ax_duration.axis("off")
		table = ax_duration.table(cellText=duration_table.values, colLabels=duration_table.columns, cellLoc="left", colLoc="left", loc="center")
		table.auto_set_font_size(False)
		table.set_fontsize(9.2)
		table.scale(1.0, 1.45)
		for row_idx in range(len(duration_table) + 1):
			for col_idx in range(len(duration_table.columns)):
				cell = table[row_idx, col_idx]
				cell.set_edgecolor("#dddddd")
				if row_idx == 0:
					cell.set_facecolor(COLOR["neutral"])
					cell.set_text_props(weight="bold")
				elif col_idx == 0:
					regime_name = duration_table.iloc[row_idx - 1, 0]
					cell.set_facecolor(regime_light.get(regime_name, "white"))
		ax_duration.set_title("Panel B. Observaciones y duración de episodios", fontsize=12, pad=8)

	if VIS_REGIME_PERF is None or VIS_REGIME_PERF.empty:
		ax_perf.text(0.5, 0.5, "Sin performance condicional", ha="center", va="center", fontsize=11)
		ax_perf.axis("off")
	else:
		perf = VIS_REGIME_PERF.copy()
		perf["Estrategia"] = perf["strategy"].map(vis_h_strategy)
		perf["Régimen"] = perf["regime_name"].map(vis_h_regime)
		wanted = ["MinVar", "MinVar sin crypto", "CVaR", "Overlay combinado"]
		perf = perf.loc[perf["Estrategia"].isin(wanted)]
		x_positions = {"Riesgo contenido": 0, "Estrés alto": 1}
		for strategy in [name for name in wanted if name in perf["Estrategia"].unique()]:
			vals = []
			for regime_name in ["Riesgo contenido", "Estrés alto"]:
				value = perf.loc[(perf["Estrategia"] == strategy) & (perf["Régimen"] == regime_name), "sharpe"].mean()
				vals.append(value)
			if all(pd.notna(vals)):
				ax_perf.plot([0, 1], vals, marker="o", linewidth=2.0, color=VIS_COLORS.get(strategy), label=strategy)
				ax_perf.text(1.02, vals[-1], strategy, va="center", fontsize=8.5, color=VIS_COLORS.get(strategy))
		ax_perf.set_xticks([0, 1], ["Riesgo contenido", "Estrés alto"])
		ax_perf.set_title("Panel C. Sharpe condicional por régimen", fontsize=12)
		ax_perf.set_ylabel("Sharpe")
		ax_perf.grid(True, axis="y", alpha=0.25, color=COLOR["grid"])
		ax_perf.axhline(0, color=COLOR["zero"], linewidth=0.9)

	fig.suptitle("Figura 7. Regímenes de mercado", fontsize=15)
	plt.show()

	crypto_regime_text = ""
	if VIS_REGIME_CRYPTO is not None and not VIS_REGIME_CRYPTO.empty:
		expo = VIS_REGIME_CRYPTO.copy()
		expo["Estrategia"] = expo["strategy"].map(vis_h_strategy)
		expo["Régimen"] = expo["regime_name"].map(vis_h_regime)
		mv_expo = expo.loc[expo["Estrategia"] == "MinVar"]
		if not mv_expo.empty:
			pieces = [f"{row.Régimen}: {row.mean_crypto_weight * 100:.2f}%" for row in mv_expo.itertuples()]
			crypto_regime_text = " Exposición crypto media MinVar por régimen: " + "; ".join(pieces) + "."

	vis_show_reading(
		"Volatilidad realizada SPY 63d como indicador observable, bandas de riesgo contenido/estrés alto, duración de episodios y Sharpe condicional.",
		"Las bandas rojas ubican episodios de estrés y las verdes periodos de riesgo contenido; el rendimiento condicional cae en estrés." + crypto_regime_text,
		"Los regímenes muestran cuándo el mercado entra en estrés y cómo cambia el rendimiento de las estrategias, pero son diagnóstico histórico, no señal live validada.",
		"No demuestra timing ex ante; una mejora futura razonable sería validar reglas de transición con datos posteriores no usados en la definición de estados.",
	)

### Figura 8 — Modelos supervisados: señal predictiva y valor económico

**Pregunta de investigación.** ¿Los modelos supervisados predicen riesgos útiles y esa señal se traduce en una mejora económica de cartera?

**Cómo se implementó**
- Código principal: `src/supervised_targets.py`, `src/supervised_features.py`, `src/supervised_validation.py`, `src/supervised_models.py`, `src/model_evaluation.py`
- Script: `scripts/run_chapter5.py`
- Outputs usados: `outputs/chapter5/model_scores.csv`, `outputs/chapter5/model_selection.csv`, `outputs/chapter5/forecast_bucket_analysis.csv`, `outputs/chapter5/calibration_tables.csv`
- Qué contrasta: modelos seleccionados, mejora frente a naive, buckets de forecast y calibración de eventos.
- Qué permite concluir: si hay señal predictiva y si conviene separarla del valor económico del overlay.

In [ ]:
def vis_target_label(name: str) -> str:
    mapping = {
        "target_portfolio_vol_21d": "Vol cartera 21d",
        "target_portfolio_vol_63d": "Vol cartera 63d",
        "target_spy_vol_21d": "Vol SPY 21d",
        "target_spy_vol_63d": "Vol SPY 63d",
        "target_btc_vol_21d": "Vol BTC 21d",
        "target_btc_vol_63d": "Vol BTC 63d",
        "target_portfolio_dd_21d": "DD cartera 21d",
        "target_portfolio_dd_63d": "DD cartera 63d",
        "target_portfolio_dd_event_21d": "Evento DD cartera 21d",
        "target_portfolio_dd_event_63d": "Evento DD cartera 63d",
        "target_btc_dd_event_21d": "Evento DD BTC 21d",
        "target_stress_event_21d": "Evento estrés 21d",
    }
    return mapping.get(name, name)


def vis_model_label(name: str) -> str:
    mapping = {
        "naive_rolling_vol": "Naive vol rolling",
        "ewma": "EWMA",
        "ridge": "Ridge",
        "elastic_net": "Elastic Net",
        "random_forest": "Random Forest",
        "random_forest_classifier": "Random Forest",
        "gradient_boosting_classifier": "Gradient Boosting",
        "logistic": "Logística",
        "calibrated_logistic": "Logística calibrada",
    }
    return mapping.get(name, name)

if VIS_MODEL_SCORES is None or VIS_MODEL_SCORES.empty or VIS_MODEL_SELECTION is None or VIS_MODEL_SELECTION.empty:
    vis_note("No están disponibles `model_scores.csv` o `model_selection.csv`.")
    vis_show_reading(
        "La selección de modelos, buckets de forecast y calibración de eventos.",
        "La figura no puede renderizarse porque faltan scores o selección de modelos.",
        "Sin scores fuera de muestra no conviene sobrerreivindicar valor predictivo.",
        "Los paneles de buckets y calibración requieren outputs agregados de Chapter 5.",
    )
else:
    selection = VIS_MODEL_SELECTION.copy()
    selection["Target"] = selection["target_col"].map(vis_target_label)
    selection["Modelo seleccionado"] = selection["selected_model"].map(vis_model_label)
    selection["Mejora vs naive"] = selection["improved_vs_naive"].map({True: "Sí", False: "No"})
    selection_view = selection[["Target", "task", "Modelo seleccionado", "Mejora vs naive"]].rename(columns={"task": "Tarea"}).head(12)

    fig = plt.figure(figsize=(16.5, 9.2), constrained_layout=True)
    grid = fig.add_gridspec(2, 2, height_ratios=[1.05, 1.0])
    ax_sel = fig.add_subplot(grid[0, 0])
    ax_bucket = fig.add_subplot(grid[0, 1])
    ax_cal = fig.add_subplot(grid[1, 0])
    ax_summary = fig.add_subplot(grid[1, 1])

    ax_sel.axis("off")
    table = ax_sel.table(cellText=selection_view.values, colLabels=selection_view.columns, cellLoc="left", colLoc="left", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8.7)
    table.scale(1.0, 1.25)
    for row_idx in range(len(selection_view) + 1):
        for col_idx in range(len(selection_view.columns)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor("#dddddd")
            if row_idx == 0:
                cell.set_facecolor(COLOR["neutral"])
                cell.set_text_props(weight="bold")
            elif col_idx == 3:
                cell.set_facecolor(COLOR["risk_contained_light"] if selection_view.iloc[row_idx - 1, col_idx] == "Sí" else "#f2ead3")
    ax_sel.set_title("Panel A. Señal predictiva: selección OOS", fontsize=12, pad=8)

    forecast = pd.DataFrame()
    if VIS_FORECAST_BUCKETS is not None and not VIS_FORECAST_BUCKETS.empty:
        selected_reg = VIS_MODEL_SELECTION.loc[VIS_MODEL_SELECTION["task"] == "regression", ["target_col", "selected_model"]]
        forecast = VIS_FORECAST_BUCKETS.merge(selected_reg, left_on=["target_col", "model_name"], right_on=["target_col", "selected_model"], how="inner")
        chosen_targets = [target for target in ["target_portfolio_vol_21d", "target_btc_vol_21d"] if target in forecast["target_col"].unique()]
        forecast = forecast.loc[forecast["target_col"].isin(chosen_targets)].copy()

    if forecast.empty:
        ax_bucket.text(0.5, 0.5, "Sin análisis por buckets", ha="center", va="center", fontsize=11)
        ax_bucket.axis("off")
    else:
        for target in forecast["target_col"].unique():
            sub = forecast.loc[forecast["target_col"] == target].sort_values("bucket")
            label = vis_target_label(target)
            ax_bucket.plot(sub["bucket"], sub["mean_pred"], marker="o", linewidth=2.0, color=COLOR["overlay_ml"], label=f"Predicho: {label}")
            ax_bucket.plot(sub["bucket"], sub["mean_realized"], marker="s", linewidth=1.8, linestyle="--", color=COLOR["sixty_forty"], label=f"Realizado: {label}")
        ax_bucket.set_title("Panel B. Forecast vs realizado", fontsize=12)
        ax_bucket.set_xlabel("Bucket de forecast")
        ax_bucket.set_ylabel("Media")
        ax_bucket.grid(True, alpha=0.25)
        ax_bucket.legend(frameon=False, fontsize=8)

    calibration = pd.DataFrame()
    if VIS_CALIBRATION is not None and not VIS_CALIBRATION.empty:
        selected_cls = VIS_MODEL_SELECTION.loc[VIS_MODEL_SELECTION["task"] == "classification", ["target_col", "selected_model"]]
        calibration = VIS_CALIBRATION.merge(selected_cls, left_on=["target_col", "model_name"], right_on=["target_col", "selected_model"], how="inner")
        chosen_targets = [target for target in ["target_stress_event_21d", "target_btc_dd_event_21d"] if target in calibration["target_col"].unique()]
        calibration = calibration.loc[calibration["target_col"].isin(chosen_targets)].copy()

    if calibration.empty:
        ax_cal.text(0.5, 0.5, "Sin calibración agregada", ha="center", va="center", fontsize=11)
        ax_cal.axis("off")
    else:
        ax_cal.plot([0, 1], [0, 1], color=COLOR["threshold"], linestyle="--", linewidth=1.0)
        for target in calibration["target_col"].unique():
            sub = calibration.loc[calibration["target_col"] == target].sort_values("decile")
            ax_cal.plot(sub["mean_score"], sub["event_rate"], marker="o", linewidth=2.0, label=vis_target_label(target))
        ax_cal.set_title("Panel C. Calibración de eventos", fontsize=12)
        ax_cal.set_xlabel("Probabilidad predicha")
        ax_cal.set_ylabel("Frecuencia observada")
        ax_cal.grid(True, alpha=0.25)
        ax_cal.legend(frameon=False, fontsize=8)

    improved = int(VIS_MODEL_SELECTION["improved_vs_naive"].fillna(False).sum())
    total = int(len(VIS_MODEL_SELECTION))
    overlay_uses = 0
    if VIS_OVERLAY_DECISIONS is not None and not VIS_OVERLAY_DECISIONS.empty:
        overlay_uses = int((VIS_OVERLAY_DECISIONS["reason"].fillna("none") != "none").sum())
    delta_sharpe_overlay = np.nan
    delta_es_overlay = np.nan
    if not VIS_BASELINE_ROW.empty and not VIS_OVERLAY_ROW.empty:
        delta_sharpe_overlay = vis_metric(VIS_OVERLAY_ROW, "sharpe") - vis_metric(VIS_BASELINE_ROW, "sharpe")
        delta_es_overlay = (vis_metric(VIS_OVERLAY_ROW, "es95") - vis_metric(VIS_BASELINE_ROW, "es95")) * 100.0
    summary_df = pd.DataFrame([
        ["Targets con señal OOS vs naive", f"{improved}/{total}"],
        ["Decisiones overlay con ajuste", overlay_uses],
        ["Δ Sharpe overlay vs MinVar", f"{delta_sharpe_overlay:+.3f}" if pd.notna(delta_sharpe_overlay) else "n/d"],
        ["Δ ES95 overlay vs MinVar", f"{delta_es_overlay:+.2f} p.p." if pd.notna(delta_es_overlay) else "n/d"],
        ["Lectura económica", "señal ≠ monetización"],
    ], columns=["Indicador", "Valor"])
    ax_summary.axis("off")
    table = ax_summary.table(cellText=summary_df.values, colLabels=summary_df.columns, cellLoc="left", colLoc="left", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10.2)
    table.scale(1.0, 1.5)
    for row_idx in range(len(summary_df) + 1):
        for col_idx in range(len(summary_df.columns)):
            cell = table[row_idx, col_idx]
            cell.set_edgecolor("#dddddd")
            if row_idx == 0:
                cell.set_facecolor(COLOR["neutral"])
                cell.set_text_props(weight="bold")
    ax_summary.set_title("Panel D. Prueba económica de cartera", fontsize=12, pad=8)

    fig.suptitle("Figura 8. Modelos supervisados: señal predictiva y valor económico", fontsize=15)
    plt.show()

    vis_show_reading(
        "Selección OOS, buckets de forecast, calibración y puente explícito hacia métricas económicas del overlay.",
        f"Hay señal OOS en algunos targets ({improved}/{total} mejoran al naive), pero el overlay muestra Δ Sharpe {delta_sharpe_overlay:+.3f} frente a MinVar cuando está disponible.",
        "Hay señal OOS en algunos targets, pero su monetización en cartera es limitada: no basta si el overlay no mejora Sharpe, drawdown, ES95 o Calmar.",
        "No demuestra que ML funcione como fuente de valor económico; una mejora futura razonable sería atribución evento a evento del overlay.",
    )

### Figura 9 — Overlay: decisiones y valor económico

**Pregunta de investigación.** ¿El overlay actúa de forma trazable y mejora el perfil económico neto frente a MinVar?

**Cómo se implementó**
- Código principal: `src/risk_overlay.py`, `src/overlay_backtest.py`
- Script: `scripts/run_risk_overlay.py`, `scripts/run_chapter5.py`
- Outputs usados: `outputs/chapter5/overlay_backtest_summary.csv`, `outputs/chapter5/overlay_decisions.csv`, `outputs/chapter5/overlay_weights.csv`
- Qué contrasta: motivos de activación, recorte aplicado al peso crypto y deltas de métricas frente a MinVar.
- Qué permite concluir: si la capa supervisada aporta valor económico o solo trazabilidad prudencial.

In [ ]:
overlay_target = "minvar_combined_overlay"
if VIS_OVERLAY_DECISIONS is None or VIS_OVERLAY_DECISIONS.empty or VIS_OVERLAY_SUMMARY is None or VIS_OVERLAY_SUMMARY.empty:
    vis_note("Faltan `overlay_decisions.csv` o `overlay_backtest_summary.csv`.")
    vis_show_reading(
        "Los disparadores del overlay, el recorte aplicado y su impacto económico frente a MinVar.",
        "La figura no puede renderizarse completa porque faltan outputs del capítulo 5.",
        "Sin decisiones registradas y sin resumen económico no puede juzgarse el valor del overlay.",
        "La capa supervisada solo se evalúa aquí con outputs finales, no con entrenamiento ni predicciones crudas.",
    )
else:
    decisions = VIS_OVERLAY_DECISIONS.copy()
    decisions["date"] = pd.to_datetime(decisions["date"])
    if overlay_target not in decisions["strategy"].unique():
        candidates = decisions.loc[~decisions["strategy"].isin(["minvar_baseline_corrected", "minvar_no_crypto_control"]), "strategy"]
        overlay_target = candidates.iloc[0] if not candidates.empty else decisions["strategy"].iloc[0]
    overlay_name = vis_h_strategy(overlay_target)
    overlay_path = decisions.loc[decisions["strategy"] == overlay_target].sort_values("date").copy()
    overlay_path["cut"] = (overlay_path["base_crypto_weight"] - overlay_path["adjusted_crypto_weight"]).clip(lower=0)
    activation_mask = (overlay_path["reason"].fillna("none") != "none") | overlay_path["cut"].gt(1e-8)
    activation_rate = float(activation_mask.mean()) if len(overlay_path) else np.nan
    mean_cut_active = overlay_path.loc[activation_mask, "cut"].mean() if activation_mask.any() else 0.0

    reason_map = {
        "dynamic_crypto_cap": "Límite crypto",
        "de_risking": "Reducción de riesgo",
        "dynamic_crypto_cap|de_risking": "Límite + reducción",
        "none": "Sin ajuste",
    }
    reason_counts = overlay_path.loc[activation_mask].copy()
    reason_counts["Motivo"] = reason_counts["reason"].fillna("none").map(lambda x: reason_map.get(x, x.replace("_", " ")))
    reason_counts = reason_counts.groupby("Motivo", as_index=False).size().sort_values("size", ascending=False)

    baseline_row = vis_pick_row(VIS_OVERLAY_SUMMARY, "strategy", "minvar_baseline_corrected")
    overlay_row = vis_pick_row(VIS_OVERLAY_SUMMARY, "strategy", overlay_target)

    fig = plt.figure(figsize=(16.8, 9.0), constrained_layout=True)
    grid = fig.add_gridspec(2, 2, height_ratios=[1.05, 0.95], width_ratios=[1.25, 1.0])
    ax_weights = fig.add_subplot(grid[0, 0])
    ax_cut = fig.add_subplot(grid[0, 1])
    ax_reason = fig.add_subplot(grid[1, 0])
    ax_delta = fig.add_subplot(grid[1, 1])

    ax_weights.plot(overlay_path["date"], overlay_path["base_crypto_weight"] * 100.0, linewidth=2.0, color=COLOR["minvar"], label="Peso crypto base")
    ax_weights.plot(overlay_path["date"], overlay_path["adjusted_crypto_weight"] * 100.0, linewidth=2.0, color=COLOR["overlay"], label="Peso crypto ajustado")
    ax_weights.set_title("Panel A. Peso base vs ajustado", fontsize=12)
    ax_weights.set_xlabel("Fecha")
    ax_weights.set_ylabel("Peso crypto (%)")
    ax_weights.grid(True, alpha=0.25)
    ax_weights.legend(frameon=False)

    ax_cut.plot(overlay_path["date"], overlay_path["cut"] * 100.0, linewidth=1.8, color=COLOR["overlay_stress"])
    ax_cut.fill_between(overlay_path["date"], 0, overlay_path["cut"] * 100.0, color=COLOR["high_stress_light"], alpha=0.55)
    ax_cut.set_title("Panel B. Recorte aplicado", fontsize=12)
    ax_cut.set_xlabel("Fecha")
    ax_cut.set_ylabel("Base - ajustado (p.p.)")
    ax_cut.grid(True, alpha=0.25)
    ax_cut.text(0.02, 0.95, f"Activación: {activation_rate * 100:.1f}%\nRecorte medio activo: {mean_cut_active * 100:.2f} p.p.", transform=ax_cut.transAxes, ha="left", va="top", fontsize=9, bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#dddddd"})

    if reason_counts.empty:
        ax_reason.text(0.5, 0.5, "Sin activaciones relevantes", ha="center", va="center", fontsize=11)
        ax_reason.axis("off")
    else:
        ax_reason.barh(reason_counts["Motivo"], reason_counts["size"], color=COLOR["overlay"])
        ax_reason.set_title("Panel C. Motivos de activación", fontsize=12)
        ax_reason.set_xlabel("Número de activaciones")
        ax_reason.grid(True, axis="x", alpha=0.25)

    ax_delta.axis("off")
    if baseline_row.empty or overlay_row.empty:
        ax_delta.text(0.5, 0.5, "Sin delta económico", ha="center", va="center", fontsize=11)
    else:
        calmar_delta = vis_metric(overlay_row, "calmar") - vis_metric(baseline_row, "calmar")
        cost_delta = vis_metric(overlay_row, "total_costs") - vis_metric(baseline_row, "total_costs")
        delta_table = pd.DataFrame({
            "Métrica": ["Δ Sharpe", "Δ MaxDD", "Δ ES95", "Δ Calmar", "Δ costes"],
            "Delta": [
                f"{vis_metric(overlay_row, 'sharpe') - vis_metric(baseline_row, 'sharpe'):+.3f}",
                f"{(vis_metric(overlay_row, 'max_drawdown') - vis_metric(baseline_row, 'max_drawdown')) * 100.0:+.2f} p.p.",
                f"{(vis_metric(overlay_row, 'es95') - vis_metric(baseline_row, 'es95')) * 100.0:+.2f} p.p.",
                f"{calmar_delta:+.3f}" if pd.notna(calmar_delta) else "n/d",
                f"{cost_delta * 100.0:+.2f} p.p." if pd.notna(cost_delta) else "n/d",
            ],
        })
        table = ax_delta.table(cellText=delta_table.values, colLabels=delta_table.columns, cellLoc="left", colLoc="left", loc="center")
        table.auto_set_font_size(False)
        table.set_fontsize(10.1)
        table.scale(1.0, 1.45)
        for row_idx in range(len(delta_table) + 1):
            for col_idx in range(len(delta_table.columns)):
                cell = table[row_idx, col_idx]
                cell.set_edgecolor("#dddddd")
                if row_idx == 0:
                    cell.set_facecolor(COLOR["neutral"])
                    cell.set_text_props(weight="bold")
        top_cuts = overlay_path.loc[overlay_path["cut"].gt(0.0005), ["date", "cut", "reason"]].sort_values("cut", ascending=False).head(3)
        if not top_cuts.empty:
            top_text = "Mayores recortes visibles: " + "; ".join(f"{row.date.date()}: {row.cut * 100:.2f} p.p." for row in top_cuts.itertuples())
            ax_delta.text(0.0, 0.02, top_text, transform=ax_delta.transAxes, ha="left", va="bottom", fontsize=8.8, color="#555555")
    ax_delta.set_title("Panel D. Deltas económicos vs MinVar", fontsize=12, pad=8)

    fig.suptitle("Figura 9. Overlay: decisiones y valor económico", fontsize=15)
    plt.show()

    vis_show_reading(
        "Peso crypto base y ajustado, recorte por fecha, motivos de activación y deltas económicos frente a MinVar.",
        f"El {overlay_name.lower()} se activa en {activation_rate * 100:.1f}% de observaciones relevantes; cuando actúa, el recorte medio es {mean_cut_active * 100:.2f} p.p.",
        "El overlay es trazable y prudencial, pero su valor económico neto es limitado si no mejora Sharpe, drawdown, ES95, Calmar o costes.",
        "No demuestra causalidad evento a evento; una mejora futura razonable sería descomponer la contribución de cada activación al retorno posterior.",
    )

### Nota metodológica secundaria — Trazabilidad de la auditoría

La comparación before/after se conserva solo como trazabilidad breve. El bloque visual principal usa exclusivamente la metodología final del proyecto.

In [ ]:
if VIS_AUDIT is None or VIS_AUDIT.empty:
    vis_note("No está disponible `audit_fix_comparison.csv`; se omite la tabla secundaria de trazabilidad metodológica.")
else:
    before = VIS_AUDIT.loc[VIS_AUDIT["scenario"].str.contains("before", case=False, na=False)].head(1)
    after = VIS_AUDIT.loc[VIS_AUDIT["scenario"].str.contains("after", case=False, na=False)].head(1)

    if before.empty or after.empty:
        vis_note("El CSV de auditoría existe, pero no identifica con claridad los escenarios before/after.")
    else:
        trace = pd.DataFrame([
            {"Métrica": "Filas de fin de semana", "Antes": int(before.iloc[0]["n_weekend_rows"]), "Final": int(after.iloc[0]["n_weekend_rows"])},
            {"Métrica": "Sharpe", "Antes": f"{float(before.iloc[0]['sharpe']):.3f}", "Final": f"{float(after.iloc[0]['sharpe']):.3f}"},
            {"Métrica": "ES95", "Antes": f"{float(before.iloc[0]['es95']) * 100.0:.2f}%", "Final": f"{float(after.iloc[0]['es95']) * 100.0:.2f}%"},
            {"Métrica": "Peso crypto medio", "Antes": f"{float(before.iloc[0]['average_crypto_weight']) * 100.0:.2f}%", "Final": f"{float(after.iloc[0]['average_crypto_weight']) * 100.0:.2f}%"},
            {"Métrica": "Rebalanceos con crypto > 2%", "Antes": int(before.iloc[0]['rebalances_crypto_gt_2pct']), "Final": int(after.iloc[0]['rebalances_crypto_gt_2pct'])},
        ])
        vis_note("Esta tabla se mantiene solo como trazabilidad metodológica. Todas las figuras principales usan el baseline final.")
        display(trace.style.hide(axis="index"))

> **Nota:** Esta tabla se mantiene solo como trazabilidad metodológica. Todas las figuras principales usan el baseline final.

Métrica,Antes,Final
Filas de fin de semana,869,0
Sharpe,1.141,1.096
ES95,1.34%,1.55%
Peso crypto medio,0.76%,0.53%
Rebalanceos con crypto > 2%,14,6


### Figura 10 — Dashboard final de evidencia

**Pregunta de investigación.** ¿Qué afirmaciones quedan defendibles después de separar construcción MinVar, efecto crypto, CVaR, regímenes y overlay?

**Cómo se implementó**
- Código principal: síntesis de outputs finales de bloques empíricos previos.
- Script: no ejecuta scripts; consume artefactos generados por los scripts de capítulos.
- Outputs usados: `data/processed/backtest_summary.csv`, `data/processed/benchmark_summary.csv`, `data/processed/robustness/confidence_summary.csv`, `data/processed/tail_risk/tail_risk_summary_net.csv`, `data/processed/regime_analysis/regime_conditional_performance_net.csv`, `outputs/chapter5/overlay_backtest_summary.csv`
- Qué contrasta: veredictos principales y evidencia numérica separada para evitar solapamiento visual.
- Qué permite concluir: el cierre de la tesis empírica sin afirmaciones absolutas.

In [ ]:
rows = []

bench_6040 = vis_pick_row(VIS_BENCH, "benchmark", "sixty_forty")
backtest_minvar = vis_pick_row(VIS_BACKTEST, "strategy", "min_variance")
if not bench_6040.empty and not backtest_minvar.empty:
	evidence_minvar = f"Sharpe {vis_metric(backtest_minvar, 'sharpe'):.2f} vs {vis_metric(bench_6040, 'sharpe'):.2f}; retorno {vis_metric(backtest_minvar, 'ann_return') * 100.0:.1f}% vs {vis_metric(bench_6040, 'ann_return') * 100.0:.1f}%."
else:
	evidence_minvar = "Comparación no disponible."
rows.append(["MinVar mejora al 60/40", "Favorable", evidence_minvar, "La mejora pertenece primero a la construcción MinVar, no a crypto por atribución automática."])

crypto_evidence = "Control sin crypto no disponible."
if not VIS_BASELINE_ROW.empty and not VIS_NO_CRYPTO_ROW.empty:
	delta_sharpe = vis_metric(VIS_BASELINE_ROW, "sharpe") - vis_metric(VIS_NO_CRYPTO_ROW, "sharpe")
	crypto_evidence = f"Delta Sharpe {delta_sharpe:+.3f}"
	if VIS_CONFIDENCE is not None and not VIS_CONFIDENCE.empty:
		anchor = VIS_CONFIDENCE.loc[VIS_CONFIDENCE["comparison_id"] == "C1_anchor_pair"]
		if not anchor.empty:
			crypto_evidence = f"Delta Sharpe {delta_sharpe:+.3f}; IC [{float(anchor.iloc[0]['ci_lower']):+.3f}, {float(anchor.iloc[0]['ci_upper']):+.3f}]"
rows.append(["Crypto no explica robustamente la mejora", "Negativa", crypto_evidence, "Un delta pequeño con intervalo que cruza cero no respalda contribución estructural de BTC/ETH."])

crypto_weight_evidence = "Sin panel de pesos."
if not VIS_CRYPTO_PANEL.empty:
	focus = VIS_CRYPTO_PANEL.loc[VIS_CRYPTO_PANEL["Estrategia"] == "MinVar", "crypto_total"]
	if not focus.empty:
		crypto_weight_evidence = f"Peso medio {focus.mean() * 100.0:.2f}%; >2% en {(focus > 0.02).mean() * 100.0:.1f}%"
rows.append(["No hay evidencia estructural", "Negativa", crypto_weight_evidence, "La exposición no aparece con frecuencia y tamaño suficientes para llamarla asignación estructural."])
rows.append(["Compatible con exposición táctica", "Mixta", crypto_weight_evidence, "BTC/ETH pueden aparecer de forma intermitente, sin que eso implique tesis estratégica."])

cvar_evidence = "Comparación no disponible."
if VIS_TAIL_NET is not None and not VIS_TAIL_NET.empty:
	cvar_row = vis_pick_row(VIS_TAIL_NET, "strategy", "cvar_baseline")
	minvar_row = vis_pick_row(VIS_TAIL_NET, "strategy", "minvar_baseline_ch1")
	if not cvar_row.empty and not minvar_row.empty:
		cvar_evidence = f"Sharpe {vis_metric(cvar_row, 'sharpe_net'):.3f} vs {vis_metric(minvar_row, 'sharpe_net'):.3f}; ES95 {vis_metric(cvar_row, 'expected_shortfall_net') * 100.0:.2f}% vs {vis_metric(minvar_row, 'expected_shortfall_net') * 100.0:.2f}%"
rows.append(["CVaR no cambia la conclusión", "Mixta", cvar_evidence, "La prueba de cola no rescata la tesis crypto estructural si ES95/MaxDD no mejoran claramente."])

regime_evidence = "Performance por régimen no disponible."
if VIS_REGIME_PERF is not None and not VIS_REGIME_PERF.empty:
	mv_reg = VIS_REGIME_PERF.loc[VIS_REGIME_PERF["strategy"] == "minvar_baseline_ch1"]
	low = mv_reg.loc[mv_reg["regime_name"] == "Low-stress / Risk-on", "sharpe"]
	high = mv_reg.loc[mv_reg["regime_name"] == "High-stress / Risk-off", "sharpe"]
	if not low.empty and not high.empty:
		regime_evidence = f"Sharpe {float(low.iloc[0]):.2f} riesgo contenido vs {float(high.iloc[0]):.2f} estrés"
rows.append(["Regímenes contextualizan", "Favorable", regime_evidence, "Ayudan a leer cuándo cambia el riesgo, pero no son una señal live validada."])

overlay_evidence = "Comparación no disponible."
if not VIS_BASELINE_ROW.empty and not VIS_OVERLAY_ROW.empty:
	delta_sharpe_overlay = vis_metric(VIS_OVERLAY_ROW, "sharpe") - vis_metric(VIS_BASELINE_ROW, "sharpe")
	delta_es_overlay = vis_metric(VIS_OVERLAY_ROW, "es95") - vis_metric(VIS_BASELINE_ROW, "es95")
	overlay_evidence = f"Delta Sharpe {delta_sharpe_overlay:+.3f}; Delta ES95 {delta_es_overlay * 100.0:+.2f} p.p."
rows.append(["Overlay no monetiza robustamente", "Mixta", overlay_evidence, "La capa es trazable y prudencial, pero debe mejorar métricas netas para tener valor económico claro."])

dashboard = pd.DataFrame(rows, columns=["Afirmación", "Veredicto", "Evidencia numérica", "Lectura"])
verdict_colors = {"Favorable": COLOR["risk_contained_light"], "Negativa": COLOR["high_stress_light"], "Mixta": "#f2ead3"}
verdict_edges = {"Favorable": COLOR["positive"], "Negativa": COLOR["negative"], "Mixta": COLOR["mixed"]}

fig, ax_cards = plt.subplots(figsize=(16.0, 7.2))
ax_cards.axis("off")
card_rows = dashboard.copy()
x_positions = [0.18, 0.50, 0.82]
y_positions = [0.72, 0.42, 0.12]
for idx, row in enumerate(card_rows.itertuples(index=False)):
	x = x_positions[idx % 3]
	y = y_positions[idx // 3]
	face = verdict_colors.get(row.Veredicto, COLOR["neutral"])
	edge = verdict_edges.get(row.Veredicto, COLOR["threshold"])
	box = FancyBboxPatch((x - 0.135, y), 0.27, 0.20, boxstyle="round,pad=0.02,rounding_size=0.025", linewidth=1.2, edgecolor=edge, facecolor=face)
	ax_cards.add_patch(box)
	ax_cards.text(x, y + 0.132, row.Afirmación, ha="center", va="center", fontsize=10.8, weight="bold", wrap=True)
	ax_cards.text(x, y + 0.060, row.Veredicto, ha="center", va="center", fontsize=11.2, weight="bold", color="#333333")
ax_cards.set_xlim(0, 1)
ax_cards.set_ylim(0, 1)
ax_cards.set_title("Figura 10. Veredictos principales", fontsize=15, pad=12)
plt.show()

display(Markdown("### Tabla 6 — Evidencia numérica final"))
display(dashboard[["Afirmación", "Evidencia numérica", "Lectura"]].style.hide(axis="index"))

vis_show_reading(
	"Tarjetas de veredicto separadas de la evidencia numérica para evitar texto cortado y solapamientos.",
	"El cierre visual mantiene la secuencia: MinVar sí; crypto estructural no; CVaR no rescata; regímenes contextualizan; overlay no monetiza de forma robusta.",
	"La evidencia principal es metodológicamente defendible, no universal: está acotada por muestra, universo, costes y especificación.",
	"La tabla final resume números clave; no reemplaza las figuras anteriores ni sus cautelas metodológicas.",
)

## 12. Limitaciones

Las conclusiones deben leerse como evidencia aplicada bajo una especificación concreta, no como resultado universal. Esta sección explicita los puntos débiles que condicionan la interpretación.

In [ ]:
limitations_text = """
- **Muestra y periodo.** La ventana OOS cubre un periodo relevante, pero sigue siendo limitada para activos con ciclos extremos como BTC y ETH.
- **Universo de activos.** El resultado depende del conjunto de activos incluido; otros universos podrían cambiar correlaciones, restricciones y oportunidades de diversificación.
- **Costes y liquidez.** El modelo incorpora costes explícitos, pero no modela toda la microestructura, slippage variable, impacto de mercado o restricciones de capacidad.
- **Estimación de riesgo.** MinVar, CVaR y regímenes dependen de ventanas, covarianzas, definición de cola y estabilidad de parámetros.
- **Bootstrap.** Los intervalos por bloques capturan parte de la dependencia temporal, pero no eliminan riesgo de especificación ni de múltiples comparaciones.
- **Regímenes.** Los estados de mercado son diagnósticos históricos. No se validan aquí como señal live de asignación.
- **Modelos supervisados.** Una mejora predictiva en targets de riesgo no garantiza valor económico neto. El overlay debe evaluarse por métricas finales de cartera.
- **Alcance de la conclusión crypto.** La evidencia rechaza una asignación estructural bajo este framework; no prueba que BTC/ETH nunca puedan tener valor táctico bajo otras reglas.
"""

display(Markdown(limitations_text))


- **Muestra y periodo.** La ventana OOS cubre un periodo relevante, pero sigue siendo limitada para activos con ciclos extremos como BTC y ETH.
- **Universo de activos.** El resultado depende del conjunto de activos incluido; otros universos podrían cambiar correlaciones, restricciones y oportunidades de diversificación.
- **Costes y liquidez.** El modelo incorpora costes explícitos, pero no modela toda la microestructura, slippage variable, impacto de mercado o restricciones de capacidad.
- **Estimación de riesgo.** MinVar, CVaR y regímenes dependen de ventanas, covarianzas, definición de cola y estabilidad de parámetros.
- **Bootstrap.** Los intervalos por bloques capturan parte de la dependencia temporal, pero no eliminan riesgo de especificación ni de múltiples comparaciones.
- **Regímenes.** Los estados de mercado son diagnósticos históricos. No se validan aquí como señal live de asignación.
- **Modelos supervisados.** Una mejora predictiva en targets de riesgo no garantiza valor económico neto. El overlay debe evaluarse por métricas finales de cartera.
- **Alcance de la conclusión crypto.** La evidencia rechaza una asignación estructural bajo este framework; no prueba que BTC/ETH nunca puedan tener valor táctico bajo otras reglas.


## 13. Conclusión final

La conclusión distingue explícitamente cinco capas: construcción MinVar, permiso a crypto, objetivo de cola, diagnóstico por regímenes y overlay supervisado.

In [ ]:
conclusion_text = """
La evidencia favorece una conclusión positiva para la construcción risk-based y negativa para la tesis estructural de crypto. En la ventana evaluada, MinVar mejora el perfil retorno-riesgo frente al 60/40 tradicional. Sin embargo, esa mejora no se explica limpiamente por BTC/ETH: el control sin crypto queda muy cerca, el delta de Sharpe atribuible a crypto es pequeño y el intervalo bootstrap cruza cero.

La exposición efectiva refuerza esta lectura. Una asignación estructural debería aparecer con frecuencia y tamaño económico material. En cambio, BTC/ETH aparecen con pesos bajos, mediana cercana a cero y episodios escasos por encima de 2%.

CVaR, regímenes y modelos supervisados aportan pruebas adicionales, no una reversión de la tesis. CVaR permite mirar riesgo de cola, pero no domina claramente a MinVar. Los regímenes muestran que el rendimiento depende del estado de mercado, pero funcionan como diagnóstico histórico, no como regla predictiva validada. Los modelos supervisados capturan señal en varios targets de riesgo, aunque esa señal no se traduce de forma robusta en valor económico neto del overlay.

Bajo este framework OOS, con costes, control sin crypto, bootstrap, CVaR, regímenes y overlay, BTC/ETH no justifican una asignación estructural. Su papel defendible es, como máximo, táctico, de baja intensidad y dependiente del contexto.
"""

display(Markdown(conclusion_text))


La evidencia favorece una conclusión positiva para la construcción risk-based y negativa para la tesis estructural de crypto. En la ventana evaluada, MinVar mejora el perfil retorno-riesgo frente al 60/40 tradicional. Sin embargo, esa mejora no se explica limpiamente por BTC/ETH: el control sin crypto queda muy cerca, el delta de Sharpe atribuible a crypto es pequeño y el intervalo bootstrap cruza cero.

La exposición efectiva refuerza esta lectura. Una asignación estructural debería aparecer con frecuencia y tamaño económico material. En cambio, BTC/ETH aparecen con pesos bajos, mediana cercana a cero y episodios escasos por encima de 2%.

CVaR, regímenes y modelos supervisados aportan pruebas adicionales, no una reversión de la tesis. CVaR permite mirar riesgo de cola, pero no domina claramente a MinVar. Los regímenes muestran que el rendimiento depende del estado de mercado, pero funcionan como diagnóstico histórico, no como regla predictiva validada. Los modelos supervisados capturan señal en varios targets de riesgo, aunque esa señal no se traduce de forma robusta en valor económico neto del overlay.

Bajo este framework OOS, con costes, control sin crypto, bootstrap, CVaR, regímenes y overlay, BTC/ETH no justifican una asignación estructural. Su papel defendible es, como máximo, táctico, de baja intensidad y dependiente del contexto.


## 14. Resumen para publicación académica/divulgativa

Esta versión resume la contribución del estudio en una página: pregunta, método, evidencia principal, cautelas y conclusión.

In [ ]:
publication_summary = """
**Crypto-Enhanced Multi-Asset Portfolio Optimization: A Quantitative Risk-Based Approach** evalúa si BTC y ETH aportan beneficios robustos de diversificación dentro de una cartera multi-activo optimizada por riesgo.

El diseño empírico compara MinVar contra el 60/40 tradicional, añade un control sin crypto, incorpora costes y valida la incertidumbre con bootstrap. Después somete la tesis crypto a tres pruebas adicionales: optimización CVaR, regímenes de mercado y overlay supervisado de riesgo.

El resultado principal es claro: MinVar mejora el perfil de la cartera frente al 60/40, pero el permiso a BTC/ETH no explica robustamente esa mejora. La diferencia incremental frente al control sin crypto es pequeña, el intervalo bootstrap cruza cero y la exposición media a crypto es baja e intermitente.

La optimización de cola, los regímenes y el overlay supervisado no cambian materialmente la conclusión. CVaR no domina de forma clara a MinVar; los regímenes contextualizan el riesgo, pero no validan una regla live; y la señal predictiva de los modelos no se convierte de forma robusta en mejora económica neta.

La contribución del proyecto es separar tres ideas que suelen mezclarse: que MinVar puede aportar valor, que crypto puede aparecer tácticamente, y que eso no equivale a justificar una asignación estructural a BTC/ETH.
"""

display(Markdown(publication_summary))


**Crypto-Enhanced Multi-Asset Portfolio Optimization: A Quantitative Risk-Based Approach** evalúa si BTC y ETH aportan beneficios robustos de diversificación dentro de una cartera multi-activo optimizada por riesgo.

El diseño empírico compara MinVar contra el 60/40 tradicional, añade un control sin crypto, incorpora costes y valida la incertidumbre con bootstrap. Después somete la tesis crypto a tres pruebas adicionales: optimización CVaR, regímenes de mercado y overlay supervisado de riesgo.

El resultado principal es claro: MinVar mejora el perfil de la cartera frente al 60/40, pero el permiso a BTC/ETH no explica robustamente esa mejora. La diferencia incremental frente al control sin crypto es pequeña, el intervalo bootstrap cruza cero y la exposición media a crypto es baja e intermitente.

La optimización de cola, los regímenes y el overlay supervisado no cambian materialmente la conclusión. CVaR no domina de forma clara a MinVar; los regímenes contextualizan el riesgo, pero no validan una regla live; y la señal predictiva de los modelos no se convierte de forma robusta en mejora económica neta.

La contribución del proyecto es separar tres ideas que suelen mezclarse: que MinVar puede aportar valor, que crypto puede aparecer tácticamente, y que eso no equivale a justificar una asignación estructural a BTC/ETH.


## 15. Referencias

Lista breve de referencias metodológicas y artefactos internos usados para situar la nota.

In [ ]:
references_text = """
**Referencias metodológicas**

- Markowitz, H. (1952). Portfolio Selection. *Journal of Finance*.
- Rockafellar, R. T. and Uryasev, S. (2000). Optimization of Conditional Value-at-Risk. *Journal of Risk*.
- Ledoit, O. and Wolf, M. (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*.
- DeMiguel, V., Garlappi, L. and Uppal, R. (2009). Optimal versus naive diversification. *Review of Financial Studies*.
- Michaud, R. O. (1989). The Markowitz optimization enigma: Is optimized optimal? *Financial Analysts Journal*.
- Hamilton, J. D. (1989). A new approach to the economic analysis of nonstationary time series and the business cycle. *Econometrica*.
- Ang, A. and Bekaert, G. (2002). International asset allocation with regime shifts. *Review of Financial Studies*.
- Lo, A. W. (2002). The statistics of Sharpe ratios. *Financial Analysts Journal*.
- Bailey, D. H., Borwein, J. M., Lopez de Prado, M. and Zhu, Q. J. (2014). The probability of backtest overfitting.
- Platanakis, E. and Urquhart, A. (2020). Should investors include Bitcoin in their portfolios? A portfolio theory approach. *British Accounting Review*.
- Liu, Y. and Tsyvinski, A. (2021). Risks and returns of cryptocurrency. *Review of Financial Studies*.

**Artefactos internos**

- Notebooks 01-05 del proyecto: backtest, robustez, tail risk, regímenes y overlay supervisado.
- Outputs finales en `data/processed/` y `outputs/chapter5/` consumidos por este notebook.
- Documentación metodológica interna en `docs/methodology.md` y notas de auditoría en `docs/audit_fixes.md`.
"""

display(Markdown(references_text))


**Referencias metodológicas**

- Markowitz, H. (1952). Portfolio Selection. *Journal of Finance*.
- Rockafellar, R. T. and Uryasev, S. (2000). Optimization of Conditional Value-at-Risk. *Journal of Risk*.
- Ledoit, O. and Wolf, M. (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*.
- DeMiguel, V., Garlappi, L. and Uppal, R. (2009). Optimal versus naive diversification. *Review of Financial Studies*.
- Michaud, R. O. (1989). The Markowitz optimization enigma: Is optimized optimal? *Financial Analysts Journal*.
- Hamilton, J. D. (1989). A new approach to the economic analysis of nonstationary time series and the business cycle. *Econometrica*.
- Ang, A. and Bekaert, G. (2002). International asset allocation with regime shifts. *Review of Financial Studies*.
- Lo, A. W. (2002). The statistics of Sharpe ratios. *Financial Analysts Journal*.
- Bailey, D. H., Borwein, J. M., Lopez de Prado, M. and Zhu, Q. J. (2014). The probability of backtest overfitting.
- Platanakis, E. and Urquhart, A. (2020). Should investors include Bitcoin in their portfolios? A portfolio theory approach. *British Accounting Review*.
- Liu, Y. and Tsyvinski, A. (2021). Risks and returns of cryptocurrency. *Review of Financial Studies*.

**Artefactos internos**

- Notebooks 01-05 del proyecto: backtest, robustez, tail risk, regímenes y overlay supervisado.
- Outputs finales en `data/processed/` y `outputs/chapter5/` consumidos por este notebook.
- Documentación metodológica interna en `docs/methodology.md` y notas de auditoría en `docs/audit_fixes.md`.
